# All models comparaison

This notebook launches five complete independent training runs:

```python
SEEDS = (42, 43, 44, 45, 46)
```
Every reported metric is aggregated as **mean ± one standard deviation**. The seed-specific results are in

```text
data/treated/spx_full_multiseed/
``` 

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import norm

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

PAPER_FONT_SIZE = 9

PAPER_RC = {
    "font.family": "serif",
    "font.serif": [
        "Linux Libertine O",
        "Libertinus Serif",
        "Times New Roman",
        "Times",
        "Nimbus Roman",
        "Liberation Serif",
        "DejaVu Serif",
    ],
    "font.size": PAPER_FONT_SIZE,
    "axes.titlesize": PAPER_FONT_SIZE,
    "axes.labelsize": PAPER_FONT_SIZE,
    "xtick.labelsize": PAPER_FONT_SIZE,
    "ytick.labelsize": PAPER_FONT_SIZE,
    "legend.fontsize": PAPER_FONT_SIZE,
    "figure.titlesize": PAPER_FONT_SIZE,
    "mathtext.fontset": "stix",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "--",
    "grid.linewidth": 0.5,
    "lines.linewidth": 1.35,
    "lines.markersize": 4.0,
    "legend.frameon": True,
    "legend.framealpha": 0.95,
    "legend.edgecolor": "0.75",
    "legend.handlelength": 2.0,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}

mpl.rcParams.update(PAPER_RC)
sns.set_theme(
    context="paper",
    style="ticks",
    rc=PAPER_RC,
)


from src.other_models.diffusion_cond_ext import fit_normalizer, normalize, denormalize
from src.volsurface_latentFM.ivs_autoencoder import IVS_AE
from src.volsurface_latentFM.latent_models_variants import LatentFlowMatch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_ROOT = Path.cwd()
CUBE_PATH = Path("data/treated/vol_surface_spx_long.npz")
if not CUBE_PATH.exists():
    raise FileNotFoundError(
        f"Expected {CUBE_PATH}. Build it first with constructor_surface_cube.py."
    )

data = np.load(CUBE_PATH, allow_pickle=True)
cube_raw = np.asarray(data["cube"], dtype=np.float32)
m_grid = np.asarray(data["m_grid"], dtype=float).ravel()
tenor_days = np.asarray(data["tenor_days"], dtype=float).ravel()
tau_grid = tenor_days / 365.0
asof_dates = np.asarray(data["asof_dates"]) if "asof_dates" in data.files else None


def _cube_to_surfaces_t_m(cube, m_grid, tenor_days):
    """Return surfaces as (dates, tenor, moneyness)."""
    cube = np.asarray(cube, dtype=np.float32)
    n_m, n_t = len(m_grid), len(tenor_days)
    if cube.ndim != 3:
        raise ValueError(f"cube must be 3D, got {cube.shape}")
    if cube.shape[0] == n_m and cube.shape[1] == n_t:      # canonical (M,T,D)
        return cube.transpose(2, 1, 0)
    if cube.shape[0] == n_t and cube.shape[1] == n_m:      # (T,M,D)
        return cube.transpose(2, 0, 1)
    if cube.shape[1] == n_t and cube.shape[2] == n_m:      # (D,T,M)
        return cube
    if cube.shape[1] == n_m and cube.shape[2] == n_t:      # (D,M,T)
        return cube.transpose(0, 2, 1)
    raise ValueError(
        f"Cannot infer cube orientation for shape={cube.shape}, "
        f"len(m_grid)={n_m}, len(tenor_days)={n_t}."
    )


def _to_percent_iv(x):
    x = np.asarray(x, dtype=np.float32)
    return x * 100.0 if np.nanmedian(x) < 2.0 else x


surfaces_t_m_pct_native = _to_percent_iv(_cube_to_surfaces_t_m(cube_raw, m_grid, tenor_days))

TARGET_NT, TARGET_NM = 16, 32
source_nt, source_nm = surfaces_t_m_pct_native.shape[-2:]
with torch.no_grad():
    real_iv = F.interpolate(
        torch.tensor(
            surfaces_t_m_pct_native,
            dtype=torch.float32,
        ).unsqueeze(1),
        size=(TARGET_NT, TARGET_NM),
        mode="bilinear",
        align_corners=False,
    ).to(device)
m_new = np.linspace(
    float(np.nanmin(m_grid)),
    float(np.nanmax(m_grid)),
    TARGET_NM,
)

tau_new = np.linspace(
    float(np.nanmin(tau_grid)),
    float(np.nanmax(tau_grid)),
    TARGET_NT,
)

# Preserve all downstream variables exactly.
NT, NM = real_iv.shape[-2], real_iv.shape[-1]

m_common = m_new
tau_common = tau_new

if len(m_new) != NM or len(tau_new) != NT:
    raise ValueError(
        "Grid lengths and surface dimensions are inconsistent."
    )

print("Loaded:", CUBE_PATH)
print("Original cube shape:", cube_raw.shape)
print("Original source grid:", (source_nt, source_nm))
print("Training tensor shape:", tuple(real_iv.shape))
print("Using exact constructor grid:", False)
print("Grid convention: historical linear 16 x 32 comparison grid")
print("Moneyness grid:",m_new[[0, -1]],"points=",len(m_new))
print("Tenor grid (years):",tau_new[[0, -1]],"points=",len(tau_new))
print("Median IV (%):",float(torch.nanmedian(real_iv).detach().cpu()))


In [ ]:
# Pipeline controls + HPO cache/helpers 
import json
import importlib
import random
from pathlib import Path

from scipy.stats import norm

SEEDS = (42, 43, 44, 45, 46)
SEED = int(os.environ.get("FULL_MODELS_SEED", SEEDS[0]))
FULL_MODELS_CHILD = bool(int(os.environ.get("FULL_MODELS_CHILD", "0")))
FULL_MODELS_FRESH_RUN = bool(int(os.environ.get("FULL_MODELS_FRESH_RUN", "1")))

SEED_ROOT = Path("data/treated/spx_full_multiseed")
SEED_ROOT.mkdir(parents=True, exist_ok=True)

SEED_DIR = SEED_ROOT


def seed_path(filename, seed=SEED):
    """Insert `_seed_<seed>` before a file extension."""
    path = Path(filename)
    return SEED_ROOT / f"{path.stem}_seed_{int(seed)}{path.suffix}"


# Every seed is a fresh retraining, but only files for that seed are
# removed. Other seeds remain in the same common output directory.
if FULL_MODELS_FRESH_RUN:
    for stale_path in SEED_ROOT.glob(f"*_seed_{SEED}.*"):
        if stale_path.is_file():
            stale_path.unlink()

MULTISEED_METRIC_SEED = 123456


def set_all_seeds(seed):
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


set_all_seeds(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

# Master switches
RUN_HPO = bool(int(os.environ.get("FULL_MODELS_RUN_HPO", "0")))
USE_BEST_HP = False
ARB_WEIGHT = float(globals().get("ARB_WEIGHT", 0.6))
ARB_TOL = 1e-4
COMMON_FINAL_EPOCHS = int(globals().get("COMMON_FINAL_EPOCHS", 1200))
COMMON_HPO_EPOCHS = int(globals().get("COMMON_HPO_EPOCHS", 300))
COMMON_HPO_TRIALS = int(globals().get("COMMON_HPO_TRIALS", 12))

HPO_TRIALS = dict(globals().get("HPO_TRIALS", {
    "ae": COMMON_HPO_TRIALS,
    "fm": COMMON_HPO_TRIALS,
    "ddpm": COMMON_HPO_TRIALS,
    "volgan": COMMON_HPO_TRIALS,
    "cont": COMMON_HPO_TRIALS,
    "sbjts_pca": COMMON_HPO_TRIALS,
    "sbjts_ae": COMMON_HPO_TRIALS,
    "lightsb": COMMON_HPO_TRIALS,
}))

HPO_FAST_EPOCHS = dict(globals().get("HPO_FAST_EPOCHS", {
    "ae": COMMON_HPO_EPOCHS,
    "fm": COMMON_HPO_EPOCHS,
    "ddpm": COMMON_HPO_EPOCHS,
    "volgan": COMMON_HPO_EPOCHS,
    "sbjts_ae": COMMON_HPO_EPOCHS,
    "lightsb": COMMON_HPO_EPOCHS,
}))

HPO_CACHE_PATH = seed_path("spx_best_hparams.json")
HPO_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

if importlib.util.find_spec("optuna") is not None:
    optuna = importlib.import_module("optuna")
    OPTUNA_AVAILABLE = True
else:
    optuna = None
    OPTUNA_AVAILABLE = False




def _load_hpo_cache(path: Path):
    if path.exists():
        try:
            return json.loads(path.read_text())
        except Exception:
            return {}
    return {}


def _save_hpo_cache(path: Path, payload: dict):
    path.write_text(json.dumps(payload, indent=2, sort_keys=True))


HPO_BEST = _load_hpo_cache(HPO_CACHE_PATH)


def _set_best(model_key: str, params: dict, score: float, extra: dict | None = None):
    HPO_BEST[model_key] = {
        "params": dict(params),
        "score": float(score),
        "extra": dict(extra or {}),
    }
    _save_hpo_cache(HPO_CACHE_PATH, HPO_BEST)


def _real_fraction_surfaces_16x32():
    return (real_iv[:, 0].detach().cpu().numpy() / 100.0).astype(np.float32)


def relative_call(sigma, m, tau, r=0.0):
    """Black--Scholes call divided by spot, using ordinary moneyness m=K/S."""
    sigma = np.asarray(sigma, dtype=float)
    m = np.asarray(m, dtype=float)
    tau = np.asarray(tau, dtype=float)
    if sigma.shape != (len(tau), len(m)):
        raise ValueError(f"sigma shape {sigma.shape} != ({len(tau)}, {len(m)})")

    T, M = np.meshgrid(tau, m, indexing="ij")
    T = np.clip(T, 1e-12, None)
    vol = np.clip(sigma, 1e-8, None)

    d1 = (-M + (r + 0.5 * vol**2) * T) / (vol * np.sqrt(T))
    d2 = d1 - vol * np.sqrt(T)
    C = norm.cdf(d1) - np.exp(M - r*T) * norm.cdf(d2)
    return np.clip(C, 1e-12, None)


def compute_phi(sigma, m, tau, r=0.0):
    """Calendar, call-spread and butterfly penalties on a non-uniform grid."""
    C = relative_call(sigma, m, tau, r=r)
    dm = np.diff(m)
    dt = np.diff(tau)

    dC_dm = (C[:, 1:] - C[:, :-1]) / dm[None, :]
    p_call = np.maximum(dC_dm, 0.0).sum()

    # Convexity: the right slope must be >= the left slope.
    d2C = C[:, 2:] - 2*C[:, 1:-1] + C[:, :-2]
    p_bfly = np.maximum(-d2C, 0.0).sum()

    # Calls must be non-decreasing in maturity at fixed moneyness.
    dC_dt = (C[1:,:] - C[:-1, :]) / dt[:, None]
    p_cal = np.maximum(-dC_dt, 0.0).sum()

    return p_call, p_bfly, p_cal, p_call + p_bfly + p_cal


def batch_phi(surfaces, m, tau):
    surfaces = np.asarray(surfaces)
    result = {"call":np.zeros(len(surfaces)),
              "butterfly":np.zeros(len(surfaces)),
              "calendar":np.zeros(len(surfaces)),
              "phi":np.zeros(len(surfaces))}
    for i, sigma in enumerate(surfaces):
        c, b, cal, total = compute_phi(sigma, m, tau)
        result["call"][i] = c
        result["butterfly"][i] = b
        result["calendar"][i] = cal
        result["phi"][i] = total
    return result
  

_relative_call = relative_call
_compute_phi = lambda sigma, m, tau, r=0.0: compute_phi(sigma, m, tau, r=r)[-1]
def _batch_phi(surfaces, m, tau):
    return batch_phi(surfaces, m, tau)["phi"]
def _sliced_wasserstein(X, Y, n_proj=100, seed=0):
    X = np.asarray(X, dtype=np.float32)
    Y = np.asarray(Y, dtype=np.float32)
    rng = np.random.default_rng(seed)
    D = X.shape[1]
    dirs = rng.standard_normal((n_proj, D))
    dirs /= np.linalg.norm(dirs, axis=1, keepdims=True)

    px = np.sort(X @ dirs.T, axis=0)
    py = np.sort(Y @ dirs.T, axis=0)

    n = min(len(px), len(py))
    if len(px) != len(py):
        t = np.linspace(0, 1, n)
        tx = np.linspace(0, 1, len(px))
        ty = np.linspace(0, 1, len(py))
        px = np.array([np.interp(t, tx, px[:, j]) for j in range(n_proj)]).T
        py = np.array([np.interp(t, ty, py[:, j]) for j in range(n_proj)]).T
    return float(np.mean(np.abs(px - py)))


def _score_generated_surfaces(real_frac, gen_frac, m_vals, tau_vals, arb_weight=0.6):
    x_real = np.asarray(real_frac, dtype=np.float32)
    x_gen = np.asarray(gen_frac, dtype=np.float32)

    n = min(len(x_real), len(x_gen))
    xr = x_real[:n]
    xg = x_gen[:n]

    swd = _sliced_wasserstein(xr.reshape(n, -1), xg.reshape(n, -1), n_proj=100, seed=SEED)
    arb = float(np.mean(_batch_phi(
        xg, np.asarray(m_vals, dtype=float), np.asarray(tau_vals, dtype=float)
    )))
    score = float(arb_weight) * arb + (1.0 - float(arb_weight)) * float(swd)
    return score, {"arb": float(arb), "swd": float(swd)}


print(f"SEEDS={SEEDS} | active SEED={SEED}")
print(f"Common output directory: {SEED_ROOT}")
print(f"Current training seed: {SEED}")
print(f"RUN_HPO={RUN_HPO} | USE_BEST_HP={USE_BEST_HP} | OPTUNA_AVAILABLE={OPTUNA_AVAILABLE}")
print(f"COMMON_FINAL_EPOCHS={COMMON_FINAL_EPOCHS} | COMMON_HPO_EPOCHS={COMMON_HPO_EPOCHS} | COMMON_HPO_TRIALS={COMMON_HPO_TRIALS}")
print(f"HPO cache: {HPO_CACHE_PATH} | entries={len(HPO_BEST)}")

In [ ]:
# Full-sample retraining: AE + latent FM variants (SPX) 
import numpy as np
import torch

from src.other_models.diffusion_cond_ext import fit_normalizer, normalize, denormalize as _dn
from src.volsurface_latentFM.ivs_autoencoder import IVS_AE
from src.volsurface_latentFM.latent_models_variants import LatentFlowMatch

set_all_seeds(SEED + 100)

# Full-sample mode: train on every available surface and evaluate on this same dataset.
split = len(real_iv)

# Normalization fitted on full sample
lo_cmp, hi_cmp = fit_normalizer(real_iv[:split])
surf_train_n = normalize(real_iv[:split], lo_cmp, hi_cmp)
lo_v = float(lo_cmp.item())
hi_v = float(hi_cmp.item())

# Small chronological validation slice used only for HPO scoring
val_size = max(64, int(0.15 * split))
if split - val_size < 64:
    val_size = max(16, split // 5)
train_size = split - val_size
surf_hpo_train_n = surf_train_n[:train_size]
surf_hpo_val_n = surf_train_n[train_size:]

# Checkpoint paths (overwriting previous artifacts on purpose)
AE_CKPT = str(seed_path("ivs_ae_spx.pt"))
FM_CKPTS = {
    "linear": str(seed_path("latent_fm_linear.pt")),
    "quadratic": str(seed_path("latent_fm_quadratic.pt")),
    "vp": str(seed_path("latent_fm_vp.pt")),
    "trigo": str(seed_path("latent_fm_trigo.pt")),
}

print(f"Full-sample setup: train/eval size = {split}, lo={lo_v:.3f}, hi={hi_v:.3f}")
print(f"HPO split: train={len(surf_hpo_train_n)} val={len(surf_hpo_val_n)}")

# ── Stage 1: choose AE hyperparameters (Optuna optional) ─────────────────────
ae_arch = {
    "hidden_enc": (256, 128, 64),
    "hidden_dec": (64, 128, 256),
}
ae_fit_cfg = {
    "n_epochs": COMMON_FINAL_EPOCHS,
    "batch_size": 64,
    "lr": 2e-3,
    "warmup_epochs": 200,
    "patience": 1500,
    "print_every": 200,
}
ae_params = {
    "z_dim": 6,
    "beta": 1e-2,
    "lambda_cal": 3e-2,
    "lambda_but": 2e-3,
}

if USE_BEST_HP and "ivs_ae_spx" in HPO_BEST:
    ae_params.update(HPO_BEST["ivs_ae_spx"].get("params", {}))
    print("Using cached best AE params:", ae_params)
elif RUN_HPO and OPTUNA_AVAILABLE and len(surf_hpo_val_n) >= 16:
    print("Running Optuna for AE...")

    real_frac_hpo = _real_fraction_surfaces_16x32()[train_size:]

    def ae_objective(trial):
        cfg = {
            "z_dim": int(trial.suggest_int("z_dim", 4, 12)),
            "beta": float(trial.suggest_float("beta", 1e-4, 2e-2, log=True)),
            "lambda_cal": float(trial.suggest_float("lambda_cal", 1e-3, 5e-1, log=True)),
            "lambda_but": float(trial.suggest_float("lambda_but", 1e-3, 5e-1, log=True)),
            "batch_size": int(trial.suggest_categorical("batch_size", [32, 64, 96])),
            "lr": float(trial.suggest_float("lr", 5e-4, 3e-3, log=True)),
        }
        ae_tmp = IVS_AE(
            z_dim=cfg["z_dim"],
            hidden_enc=ae_arch["hidden_enc"],
            hidden_dec=ae_arch["hidden_dec"],
            beta=cfg["beta"],
            lambda_cal=cfg["lambda_cal"],
            lambda_but=cfg["lambda_but"],
        )
        ae_tmp.fit(
            surf_hpo_train_n,
            m_new,
            tau_new,
            surf_val_n=surf_hpo_val_n,
            n_epochs=int(HPO_FAST_EPOCHS.get("ae", 400)),
            batch_size=cfg["batch_size"],
            lr=cfg["lr"],
            warmup_epochs=80,
            patience=120,
            print_every=10**9,
            lo=lo_v,
            hi=hi_v,
            device=device,
        )
        with torch.no_grad():
            z_gen = torch.randn(min(512, len(real_frac_hpo)), cfg["z_dim"], device=device)
            x_hat = ae_tmp.decoder(z_gen).detach().cpu().numpy().reshape(-1, 16, 32)
        gen_pct = ((x_hat + 1.0) / 2.0) * (hi_v - lo_v) + lo_v
        gen_frac = (gen_pct / 100.0).astype(np.float32)

        n_eval = min(len(real_frac_hpo), len(gen_frac))
        score, parts = _score_generated_surfaces(real_frac_hpo[:n_eval], gen_frac[:n_eval], m_new, tau_new, arb_weight=ARB_WEIGHT)
        trial.set_user_attr("arb", parts["arb"])
        trial.set_user_attr("swd", parts["swd"])
        return score

    ae_study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    ae_study.optimize(ae_objective, n_trials=int(HPO_TRIALS.get("ae", 12)), show_progress_bar=False)
    ae_best = ae_study.best_trial
    ae_params.update({
        "z_dim": int(ae_best.params["z_dim"]),
        "beta": float(ae_best.params["beta"]),
        "lambda_cal": float(ae_best.params["lambda_cal"]),
        "lambda_but": float(ae_best.params["lambda_but"]),
    })
    ae_fit_cfg.update({
        "batch_size": int(ae_best.params["batch_size"]),
        "lr": float(ae_best.params["lr"]),
    })
    _set_best(
        "ivs_ae_spx",
        ae_params,
        score=float(ae_best.value),
        extra={"arb": float(ae_best.user_attrs.get("arb", np.nan)), "swd": float(ae_best.user_attrs.get("swd", np.nan)), "trial": int(ae_best.number)},
    )
    print("Selected AE params:", ae_params)
else:
    print("Using default AE params:", ae_params)

# ── Stage 1b: retrain AE on full dataset with selected params ─────────────────
set_all_seeds(SEED + 100)
ae = IVS_AE(
    z_dim=int(ae_params["z_dim"]),
    hidden_enc=ae_arch["hidden_enc"],
    hidden_dec=ae_arch["hidden_dec"],
    beta=float(ae_params["beta"]),
    lambda_cal=float(ae_params["lambda_cal"]),
    lambda_but=float(ae_params["lambda_but"]),
)
ae.fit(
    surf_train_n,
    m_new,
    tau_new,
    surf_val_n=None,
    n_epochs=int(ae_fit_cfg["n_epochs"]),
    batch_size=int(ae_fit_cfg["batch_size"]),
    lr=float(ae_fit_cfg["lr"]),
    warmup_epochs=int(ae_fit_cfg["warmup_epochs"]),
    patience=int(ae_fit_cfg["patience"]),
    print_every=int(ae_fit_cfg["print_every"]),
    lo=lo_v,
    hi=hi_v,
    device=device,
)
ae.save(AE_CKPT, surf_lo=lo_v, surf_hi=hi_v)

# ── Stage 2: choose FM hyperparameters (per path) + retrain full sample ───────
ae = IVS_AE.load(AE_CKPT, device=device)
ae.eval()
vae_lo = torch.tensor(ae.surf_lo, dtype=torch.float32, device=device)
vae_hi = torch.tensor(ae.surf_hi, dtype=torch.float32, device=device)

z_train = ae.encode(surf_train_n, device=device, use_mean=True)
z_hpo_train = ae.encode(surf_hpo_train_n, device=device, use_mean=True)

print(f"Latent train shape={z_train.shape}, mean={z_train.mean():.4f}, std={z_train.std():.4f}")


In [ ]:
# Compact latent-space diagnostics (run immediately after AE training)
import os
import numpy as np
import matplotlib.pyplot as plt

os.makedirs("figs", exist_ok=True)

z_all = np.asarray(z_train, dtype=np.float32)
np.save("data/treated/z_train_spx_full.npy", z_all)

atm_idx = int(np.argmin(np.abs(np.asarray(m_new) - 1.0)))
# A stable level indicator: ATM IV averaged across all maturities.
iv_atm_level = real_iv[:, 0, :, atm_idx].detach().cpu().numpy().mean(axis=1)
low_threshold = float(np.percentile(iv_atm_level, 25.0))
low_mask = iv_atm_level <= low_threshold

print("Latent-code diagnostics")
print(f"shape={z_all.shape} | global mean={z_all.mean():.4f} | global std={z_all.std():.4f}")
print(f"low-volatility threshold (25th pct)={low_threshold:.3f}% | count={low_mask.sum()}")
print("\nPer-dimension statistics")
print(f"{'dim':<6} {'mean':>10} {'std':>10} {'min':>10} {'max':>10}")
for d in range(z_all.shape[1]):
    v = z_all[:, d]
    print(f"z{d:<5} {v.mean():10.4f} {v.std():10.4f} {v.min():10.4f} {v.max():10.4f}")

corr = np.corrcoef(z_all, rowvar=False)
print("\nLatent correlation matrix")
print(np.round(corr, 3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
if z_all.shape[1] >= 2:
    sc = axes[0].scatter(z_all[:, 0], z_all[:, 1], c=iv_atm_level, s=14, alpha=0.75)
    axes[0].set_xlabel(r"$z_1$")
    axes[0].set_ylabel(r"$z_2$")
    axes[0].set_title("Latent space coloured by ATM IV level")
    plt.colorbar(sc, ax=axes[0], label="ATM IV level (%)")
else:
    axes[0].hist(z_all[:, 0], bins=40)
    axes[0].set_title("Latent-code distribution")

im = axes[1].imshow(corr, vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
axes[1].set_title("Latent correlation matrix")
axes[1].set_xlabel("Latent dimension")
axes[1].set_ylabel("Latent dimension")
plt.colorbar(im, ax=axes[1], label="Correlation")
plt.tight_layout()
plt.savefig("figs/latent_space_diagnostics.pdf", bbox_inches="tight")
plt.show()

print("\nLow-volatility latent mean:", np.round(z_all[low_mask].mean(axis=0), 3))
print("Low-volatility latent std :", np.round(z_all[low_mask].std(axis=0), 3))


In [ ]:
real_frac_hpo = _real_fraction_surfaces_16x32()[train_size:]

fm_objects = {}
fm_defaults = {
    "linear": {"n_epochs": COMMON_FINAL_EPOCHS, "batch_size": 64, "lr": 1e-4},
    "quadratic": {"n_epochs": COMMON_FINAL_EPOCHS, "batch_size": 64, "lr": 1e-4},
    "vp": {"n_epochs": COMMON_FINAL_EPOCHS, "batch_size": 64, "lr": 1e-4},
    "trigo": {"n_epochs": COMMON_FINAL_EPOCHS, "batch_size": 64, "lr": 1e-4},
}


def _decode(z_np):
    x_hat = ae.decode(z_np, device=device)
    surf = x_hat.reshape(-1, 16, 32).cpu()
    return _dn(surf, vae_lo.cpu(), vae_hi.cpu()).numpy()


def _tune_fm_params(path_name):
    if USE_BEST_HP and f"fm_{path_name}" in HPO_BEST:
        best = HPO_BEST[f"fm_{path_name}"]["params"]
        print(f"Using cached best FM[{path_name}] params: {best}")
        return {
            "n_epochs": int(best.get("n_epochs", HPO_FAST_EPOCHS.get("fm", 800))),
            "batch_size": int(best.get("batch_size", 64)),
            "lr": float(best.get("lr", 1e-4)),
        }

    if not (RUN_HPO and OPTUNA_AVAILABLE and len(surf_hpo_val_n) >= 16):
        return {
            "n_epochs": int(fm_defaults[path_name]["n_epochs"]),
            "batch_size": int(fm_defaults[path_name]["batch_size"]),
            "lr": float(fm_defaults[path_name]["lr"]),
        }

    print(f"Running Optuna for FM[{path_name}]...")

    def objective(trial):
        trial_cfg = {
            "n_epochs": int(trial.suggest_int("n_epochs", 400, int(HPO_FAST_EPOCHS.get("fm", 800)), step=200)),
            "batch_size": int(trial.suggest_categorical("batch_size", [32, 64, 128])),
            "lr": float(trial.suggest_float("lr", 5e-5, 5e-4, log=True)),
        }
        fm_tmp = LatentFlowMatch(z_dim=ae.z_dim, path=path_name)
        fm_tmp.fit(
            z_hpo_train,
            n_epochs=trial_cfg["n_epochs"],
            batch_size=trial_cfg["batch_size"],
            lr=trial_cfg["lr"],
            print_every=max(100, trial_cfg["n_epochs"] + 1),
            device=device,
        )
        n_eval = min(512, len(real_frac_hpo))
        gen_pct = _decode(fm_tmp.sample(n_eval, device=device))
        gen_frac = (gen_pct / 100.0).astype(np.float32)
        score, parts = _score_generated_surfaces(real_frac_hpo[:n_eval], gen_frac[:n_eval], m_new, tau_new, arb_weight=ARB_WEIGHT)
        trial.set_user_attr("arb", parts["arb"])
        trial.set_user_attr("swd", parts["swd"])
        return score

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=int(HPO_TRIALS.get("fm", 8)), show_progress_bar=False)
    best = study.best_trial
    best_cfg = {
        "n_epochs": int(best.params["n_epochs"]),
        "batch_size": int(best.params["batch_size"]),
        "lr": float(best.params["lr"]),
    }
    _set_best(
        f"fm_{path_name}",
        best_cfg,
        score=float(best.value),
        extra={"arb": float(best.user_attrs.get("arb", np.nan)), "swd": float(best.user_attrs.get("swd", np.nan)), "trial": int(best.number)},
    )
    print(f"Best FM[{path_name}] params: {best_cfg} | score={best.value:.6f}")
    return best_cfg


for path_index, path_name in enumerate(["linear", "quadratic", "vp", "trigo"]):
    set_all_seeds(SEED + 200 + path_index)
    tune_cfg = _tune_fm_params(path_name)
    final_cfg = {
        "n_epochs": int(fm_defaults[path_name]["n_epochs"]),
        "batch_size": int(tune_cfg["batch_size"]),
        "lr": float(tune_cfg["lr"]),
    }

    fm = LatentFlowMatch(z_dim=ae.z_dim, path=path_name)
    fm.fit(
        z_train,
        n_epochs=final_cfg["n_epochs"],
        batch_size=final_cfg["batch_size"],
        lr=final_cfg["lr"],
        print_every=500,
        device=device,
    )
    fm.save(FM_CKPTS[path_name])
    fm_objects[path_name] = fm

print("Saved FM checkpoints:")
for k, v in FM_CKPTS.items():
    print(f"  {k:<9} -> {v}")

# ── Decode helper + generated samples used by downstream comparison cells ────
N_GEN = 5000

set_all_seeds(SEED + 210)
ivs_ldm_fm_linear_raw = _decode(fm_objects["linear"].sample(N_GEN, device=device))

set_all_seeds(SEED + 211)
ivs_ldm_fm_quad_raw = _decode(fm_objects["quadratic"].sample(N_GEN, device=device))

set_all_seeds(SEED + 212)
ivs_ldm_fm_vp_raw = _decode(fm_objects["vp"].sample(N_GEN, device=device))

set_all_seeds(SEED + 213)
ivs_ldm_fm_trigo = _decode(fm_objects["trigo"].sample(N_GEN, device=device))

# Backward-compat aliases used in later cells
ivs_ldm_fm_raw = ivs_ldm_fm_trigo
l_fm = ivs_ldm_fm_trigo / 100.0

print(f"Generated {N_GEN} surfaces per FM variant after full-sample retraining.")
print(f"Hyperparameter cache updated at: {HPO_CACHE_PATH}")

In [ ]:
# Full-sample Latent DDPM: train, save and generate 
from src.volsurface_latentFM.latent_models import LatentDDPM

set_all_seeds(SEED + 300)

LATENT_DDPM_CKPT = str(seed_path("latent_ddpm_spx_full.pt"))
LATENT_DDPM_CFG = {
    "T": int(globals().get("LATENT_DDPM_T", 200)),
    "hidden": tuple(globals().get("LATENT_DDPM_HIDDEN", (64, 128, 64))),
    "n_epochs": int(globals().get("LATENT_DDPM_EPOCHS", COMMON_FINAL_EPOCHS)),
    "batch_size": int(globals().get("LATENT_DDPM_BATCH_SIZE", 64)),
    "lr": float(globals().get("LATENT_DDPM_LR", 1e-4)),
}

latent_ddpm = LatentDDPM(
    z_dim=ae.z_dim,
    T=LATENT_DDPM_CFG["T"],
    hidden=LATENT_DDPM_CFG["hidden"],
)
latent_ddpm.fit(
    z_train,
    n_epochs=LATENT_DDPM_CFG["n_epochs"],
    batch_size=LATENT_DDPM_CFG["batch_size"],
    lr=LATENT_DDPM_CFG["lr"],
    print_every=200,
    device=device,
)
latent_ddpm.save(LATENT_DDPM_CKPT)

set_all_seeds(SEED + 301)
z_latent_ddpm = latent_ddpm.sample(N_GEN, device=device)
ivs_latent_ddpm_raw = _decode(z_latent_ddpm)

print(
    f"Latent DDPM saved -> {LATENT_DDPM_CKPT} | "
    f"generated={ivs_latent_ddpm_raw.shape} | "
    f"range=[{ivs_latent_ddpm_raw.min():.2f}%, {ivs_latent_ddpm_raw.max():.2f}%]"
)


In [ ]:
# Density and Smile Comparison 
import seaborn as sns
from scipy.stats import norm
import matplotlib.pyplot as plt
import numpy as np

real_np   = real_iv[:, 0].cpu().numpy() / 100.0   # (vol surfaces, 16, 32)
l_fm   = ivs_ldm_fm_trigo / 100.0              # (N_GEN, 16, 32)

models = {'Real': real_np, 'L-FM': l_fm}
clr    = {'Real': 'black', 'L-FM': 'darkorange'}

# 1. Density comparison
fig, ax = plt.subplots(figsize=(8, 5))
for name, arr in models.items():
    sns.kdeplot(arr.reshape(-1), ax=ax, label=name, color=clr[name], linewidth=2)
ax.set_xlim(real_iv.min().item() / 100 * 0.9, 0.6)
#ax.set_xlim(-0.05, 0.6)
ax.set_xlabel('Implied Volatility')
ax.set_ylabel('Density')
ax.set_title('Distribution of Implied Volatility — Latent Models vs Real')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Summary statistics
print(f"\n{'Model':<12} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'Skew':>8}")
print('-' * 56)
for name, arr in models.items():
    x = arr.reshape(-1)
    skew = ((x - x.mean())**3).mean() / x.std()**3
    print(f"{name:<12} {x.mean():8.4f} {x.std():8.4f} {x.min():8.4f} {x.max():8.4f} {skew:8.4f}")

# 2. Average Smile + Skew
t_indices = [2, NT // 2, NT - 3]
t_labels  = [f'Short  τ={tau_new[i]:.2f}yr' for i in t_indices]
t_labels[1] = f'Mid  τ={tau_new[t_indices[1]]:.2f}yr'
t_labels[2] = f'Long  τ={tau_new[t_indices[2]]:.2f}yr'

# Average smile
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, t_idx, t_lbl in zip(axes, t_indices, t_labels):
    for name, arr in models.items():
        ax.plot(m_new, arr[:, t_idx, :].mean(0),
                label=name, color=clr[name], linewidth=2)
    ax.set_title(t_lbl)
    ax.set_xlabel('Moneyness')
    ax.set_ylabel('Implied Volatility')
    ax.legend(fontsize=PAPER_FONT_SIZE)
    ax.grid(alpha=0.3)
fig.suptitle('Average Volatility Smile — Latent Models vs Real', fontsize=PAPER_FONT_SIZE, y=1.02)
plt.tight_layout()
plt.show()

# Average skew dIV/dK
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, t_idx, t_lbl in zip(axes, t_indices, t_labels):
    for name, arr in models.items():
        skews = np.gradient(arr[:, t_idx, :], m_new, axis=1)
        ax.plot(m_new, skews.mean(0), label=name, color=clr[name], linewidth=2)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_title(t_lbl)
    ax.set_xlabel('Moneyness')
    ax.set_ylabel('dIV / dK')
    ax.legend(fontsize=PAPER_FONT_SIZE)
    ax.grid(alpha=0.3)
fig.suptitle('Average Volatility Skew (dIV/dK) — Latent Models vs Real', fontsize=PAPER_FONT_SIZE, y=1.02)
plt.tight_layout()
plt.show()

# Reuse the corrected no-arbitrage helpers from the pipeline-control cell.
if "batch_phi" not in globals():
    raise RuntimeError("Run the pipeline-control/helper cell first.")


pen_real = batch_phi(real_np,  m_new, tau_new)
pen_fm = batch_phi(l_fm,  m_new, tau_new)

results = {'Real': pen_real, 'L-FM': pen_fm}
COLORS  = {'Real': 'black', 'L-FM': 'darkorange'}

keys   = ['phi',           'calendar',                    'call',                  'butterfly']
labels = [r'$\Phi$ total', r'$p_1$ calendar spread', r'$p_2$ monotonicity', r'$p_3$ butterfly']

# Histograms
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, key, lbl in zip(axes, keys, labels):
    for name, pen in results.items():
        ax.hist(pen[key], bins=25, density=True, alpha=0.55,
                color=COLORS[name], label=name)
    ax.set_xlabel('Penalty'); ax.set_ylabel('Density')
    ax.set_title(lbl); ax.legend()
fig.suptitle(r'No-Arbitrage Penalty $\Phi$ — Distribution Across Surfaces', fontsize=PAPER_FONT_SIZE)
plt.tight_layout(); plt.show()

# Boxplots
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
model_names = list(results.keys())
for ax, key, lbl in zip(axes, keys, labels):
    bp = ax.boxplot([results[n][key] for n in model_names],
                    patch_artist=True, notch=False,
                    medianprops=dict(color='black', lw=2))
    for patch, name in zip(bp['boxes'], model_names):
        patch.set_facecolor(COLORS[name]); patch.set_alpha(0.65)
    ax.set_xticklabels(model_names, rotation=15)
    ax.set_ylabel('Penalty'); ax.set_title(lbl)
fig.suptitle('No-Arbitrage Penalties — Boxplot', fontsize=PAPER_FONT_SIZE)
plt.tight_layout(); plt.show()

# Stacked bar breakdown
fig, ax = plt.subplots(figsize=(7, 5))
x   = np.arange(len(model_names))
p1m = [results[n]['calendar'].mean() for n in model_names]
p2m = [results[n]['call'].mean() for n in model_names]
p3m = [results[n]['butterfly'].mean() for n in model_names]
ax.bar(x, p1m,             color='#4878CF', alpha=0.85, label=r'$p_1$ calendar')
ax.bar(x, p2m, bottom=p1m, color='#6ACC65', alpha=0.85, label=r'$p_2$ moneyness')
ax.bar(x, p3m, bottom=np.add(p1m, p2m), color='#D65F5F', alpha=0.85, label=r'$p_3$ butterfly')
ax.set_xticks(x); ax.set_xticklabels(model_names, fontsize=PAPER_FONT_SIZE)
ax.set_ylabel(r'Mean $\Phi$'); ax.set_title('Mean Penalty Breakdown by Component')
ax.legend(); plt.tight_layout(); plt.show()

# Summary table
print(f"\n{'':14s} {'Real':>12s} {'LDM-DDPM':>12s} {'LDM-FM':>12s}")
print('-' * 54)
for key, lbl in zip(keys, ['phi (total)', 'p1 calendar', 'p2 moneyness', 'p3 butterfly']):
    row = '  '.join(f"{results[n][key].mean():12.5f}" for n in model_names)
    print(f"{lbl:<16} {row}")
print()
for name in model_names:
    phi = results[name]['phi']
    n0  = (np.abs(phi) <  1e-4).sum()
    print(f"{name:<12}  zero-arb: {n0}/{len(phi)}  ({100*n0/len(phi):.1f}%)   max phi={phi.max():.5f}")


In [ ]:
# Density, Smile & Skew
import os, seaborn as sns
from scipy.stats import norm
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

os.makedirs('figs', exist_ok=True)

mpl.rcParams.update(PAPER_RC)
sns.set_theme(context="paper", style="ticks", rc=PAPER_RC)

real_np = real_iv[:, 0].cpu().numpy() / 100.0   # (1001, 16, 32)
l_fm    = ivs_ldm_fm_trigo / 100.0  # alias (trigo variant)                  # (N_GEN, 16, 32)

models = {
    'Data (target)' : real_np,
    'L-FM linear'   : ivs_ldm_fm_linear_raw / 100.0,
    'L-FM quadratic': ivs_ldm_fm_quad_raw   / 100.0,
    'L-FM VP'       : ivs_ldm_fm_vp_raw     / 100.0,
    'L-FM trigo'    : ivs_ldm_fm_trigo       / 100.0,
}
clr = {
    'Data (target)' : '#222222',
    'L-FM linear'   : '#E07B00',
    'L-FM quadratic': '#FF6B6B',
    'L-FM VP'       : '#C44E52',
    'L-FM trigo'    : '#8B0000',
}
ls = {
    'Data (target)' : '-',
    'L-FM linear'   : '--',
    'L-FM quadratic': '-.',
    'L-FM VP'       : ':',
    'L-FM trigo'    : (0, (3, 1, 1, 1)),
}

# IV Density 
fig, ax = plt.subplots(figsize=(7, 4.5))
for name, arr in models.items():
    sns.kdeplot(arr.reshape(-1), ax=ax, label=name,
                color=clr[name], linewidth=2.4, linestyle=ls[name])
ax.set_xlim(real_iv.min().item() / 100 * 0.85, 0.62)
ax.set_xlabel('Implied Volatility')
ax.set_ylabel('Density')
ax.set_title('Implied Volatility Distribution')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('figs/fig_density.pdf', bbox_inches='tight')
plt.show()

# Summary statistics
print(f"\n{'Model':<18} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'Skew':>8}")
print('-' * 66)
for name, arr in models.items():
    x    = arr.reshape(-1)
    skew = ((x - x.mean())**3).mean() / x.std()**3
    print(f"{name:<18} {x.mean():8.4f} {x.std():8.4f} {x.min():8.4f} {x.max():8.4f} {skew:8.4f}")

# Average Volatility Smile 
t_indices = [2, NT // 2, NT - 3]
t_labels  = [
    rf'Short  $\tau={tau_new[t_indices[0]]:.2f}$ yr',
    rf'Medium $\tau={tau_new[t_indices[1]]:.2f}$ yr',
    rf'Long   $\tau={tau_new[t_indices[2]]:.2f}$ yr',
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True)
for k, (ax, t_idx, t_lbl) in enumerate(zip(axes, t_indices, t_labels)):
    for name, arr in models.items():
        ax.plot(m_new, arr[:, t_idx, :].mean(0),
                label=name, color=clr[name], linewidth=2.4, linestyle=ls[name])
    ax.set_title(t_lbl)
    ax.set_xlabel('Moneyness')
    if k == 0:
        ax.set_ylabel('Implied Volatility')
    ax.legend(loc='upper right')
fig.suptitle('Average Volatility Smile', fontsize=PAPER_FONT_SIZE, y=1.03)
plt.tight_layout()
plt.savefig('figs/fig_smile.pdf', bbox_inches='tight')
plt.show()

# Average Volatility Skew
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True)
for k, (ax, t_idx, t_lbl) in enumerate(zip(axes, t_indices, t_labels)):
    for name, arr in models.items():
        skews = np.gradient(arr[:, t_idx, :], m_new, axis=1)
        ax.plot(m_new, skews.mean(0), label=name,
                color=clr[name], linewidth=2.4, linestyle=ls[name])
    ax.axhline(0, color='0.55', linestyle=':', linewidth=1.4)
    ax.set_title(t_lbl)
    ax.set_xlabel('Moneyness')
    if k == 0:
        ax.set_ylabel(r'$\partial\,\sigma_{\mathrm{IV}}\,/\,\partial K$')
    ax.legend(loc='upper right')
fig.suptitle('Average Volatility Skew', fontsize=PAPER_FONT_SIZE, y=1.03)
plt.tight_layout()
plt.savefig('figs/fig_skew.pdf', bbox_inches='tight')
plt.show()

mpl.rcParams.update(PAPER_RC)

# Arbitrage penalties 
if "batch_phi" not in globals():
    raise RuntimeError("Run the pipeline-control/helper cell first.")


pen_real = batch_phi(real_np, m_new, tau_new)
pen_fm   = batch_phi(l_fm,   m_new, tau_new)

results     = {'Data (target)': pen_real, 'L-FM': pen_fm}
COLORS      = {'Data (target)': '#222222', 'L-FM': '#E07B00'}
keys        = ['phi', 'calendar', 'call', 'butterfly']
labels      = [r'$\Phi$ total', r'$p_1$ calendar', r'$p_2$ monotonicity', r'$p_3$ butterfly']
model_names = list(results.keys())

# Histograms
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, key, lbl in zip(axes, keys, labels):
    for name, pen in results.items():
        ax.hist(pen[key], bins=25, density=True, alpha=0.55,
                color=COLORS[name], label=name)
    ax.set_xlabel('Penalty'); ax.set_ylabel('Density')
    ax.set_title(lbl); ax.legend()
fig.suptitle(r'No-Arbitrage Penalty $\Phi$ — Distribution Across Surfaces', fontsize=PAPER_FONT_SIZE)
plt.tight_layout(); plt.show()

# Boxplots
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, key, lbl in zip(axes, keys, labels):
    bp = ax.boxplot([results[n][key] for n in model_names],
                    patch_artist=True, notch=False,
                    medianprops=dict(color='black', lw=2))
    for patch, name in zip(bp['boxes'], model_names):
        patch.set_facecolor(COLORS[name]); patch.set_alpha(0.65)
    ax.set_xticklabels(model_names, rotation=15)
    ax.set_ylabel('Penalty'); ax.set_title(lbl)
fig.suptitle('No-Arbitrage Penalties — Boxplot', fontsize=PAPER_FONT_SIZE)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
x   = np.arange(len(model_names))
p1m = [results[n]['calendar'].mean()  for n in model_names]
p2m = [results[n]['call'].mean()      for n in model_names]
p3m = [results[n]['butterfly'].mean() for n in model_names]
ax.bar(x, p1m,                         color='#4878CF', alpha=0.85, label=r'$p_1$ calendar')
ax.bar(x, p2m, bottom=p1m,             color='#6ACC65', alpha=0.85, label=r'$p_2$ moneyness')
ax.bar(x, p3m, bottom=np.add(p1m,p2m), color='#D65F5F', alpha=0.85, label=r'$p_3$ butterfly')
ax.set_xticks(x); ax.set_xticklabels(model_names, fontsize=PAPER_FONT_SIZE)
ax.set_ylabel(r'Mean $\Phi$'); ax.set_title('Mean Penalty Breakdown by Component')
ax.legend(); plt.tight_layout(); plt.show()

# Summary table
print(f"\n{'':14s} {'Data (target)':>12s} {'L-FM':>12s}")
print('-' * 40)
for key, lbl in zip(keys, ['phi (total)', 'p1 calendar', 'p2 moneyness', 'p3 butterfly']):
    row = '  '.join(f"{results[n][key].mean():12.5f}" for n in model_names)
    print(f"{lbl:<16} {row}")

for name in model_names:
    phi = np.asarray(results[name]["phi"],dtype=float)
    n_zero = int(np.sum(np.abs(phi) < ARB_TOL))
    percentage = (100.0 * n_zero / len(phi))

    print(
        f"{name:<30}"
        f" zero-arb: {n_zero}/{len(phi)}"
        f" ({percentage:5.1f}%)"
        f"   mean phi={phi.mean():.6f}"
        f"   max phi={phi.max():.6f}"
    )


In [ ]:
# IV-DDPM + VolGAN
import numpy as np
import torch

from src.other_models.iv_ddpm import IVDDPM
from src.other_models.volgan_train import train_volgan, sample_volgan_diverse

if "surf_train_n" not in globals():
    raise RuntimeError("surf_train_n is missing. Run the AE/FM full-training cell first.")

N_GEN = 5000
IV_DDPM_CKPT = str(seed_path("iv_ddpm_spx_full.pt"))
VOLGAN_CKPT = str(seed_path("volgan_spx_full.pt"))
real_frac_all = _real_fraction_surfaces_16x32()

# Chronological split for HPO only
split_n = len(surf_train_n)
val_n = max(64, int(0.15 * split_n))
if split_n - val_n < 64:
    val_n = max(16, split_n // 5)
train_n = split_n - val_n
surf_hpo_train_n = surf_train_n[:train_n]
surf_hpo_val_n = surf_train_n[train_n:]
real_frac_val = real_frac_all[train_n:]


# IV-DDPM
set_all_seeds(SEED + 400)
ddpm_defaults = {
    "T": 500,
    "enc_channels": 32,
    "bot_channels": 64,
    "emb_dim": 128,
    "lambda_smile": 0.01,
    "lambda_ttm": 0.01,
    "n_epochs": COMMON_FINAL_EPOCHS,
    "batch_size": 64,
    "lr": 3e-4,
    "patience": 200,
    "ddim_steps": 100,
}

if USE_BEST_HP and "iv_ddpm" in HPO_BEST:
    ddpm_best = dict(HPO_BEST["iv_ddpm"].get("params", {}))
    ddpm_defaults.update(ddpm_best)
    print("Using cached best DDPM params:", ddpm_best)
elif RUN_HPO and OPTUNA_AVAILABLE and len(surf_hpo_val_n) >= 16:
    print("Running Optuna for IV-DDPM...")

    def ddpm_objective(trial):
        cfg = {
            "enc_channels": int(trial.suggest_categorical("enc_channels", [16, 24, 32, 48])),
            "bot_channels": int(trial.suggest_categorical("bot_channels", [32, 48, 64, 96])),
            "emb_dim": int(trial.suggest_categorical("emb_dim", [64, 96, 128, 160])),
            "lambda_smile": float(trial.suggest_float("lambda_smile", 1e-3, 5e-2, log=True)),
            "lambda_ttm": float(trial.suggest_float("lambda_ttm", 1e-3, 5e-2, log=True)),
            "lr": float(trial.suggest_float("lr", 1e-4, 7e-4, log=True)),
            "batch_size": int(trial.suggest_categorical("batch_size", [32, 64, 96])),
            "ddim_steps": int(trial.suggest_categorical("ddim_steps", [40, 60, 80, 100])),
        }
        tmp = IVDDPM(
            tau_grid=tau_new,
            lo=lo_v,
            hi=hi_v,
            T=300,
            enc_channels=cfg["enc_channels"],
            bot_channels=cfg["bot_channels"],
            emb_dim=cfg["emb_dim"],
            lambda_smile=cfg["lambda_smile"],
            lambda_ttm=cfg["lambda_ttm"],
            device=device,
        )
        tmp.fit(
            surf_hpo_train_n,
            surf_val_n=surf_hpo_val_n,
            n_epochs=int(HPO_FAST_EPOCHS.get("ddpm", 250)),
            batch_size=cfg["batch_size"],
            lr=cfg["lr"],
            print_every=10**9,
            patience=80,
        )
        n_eval = min(512, len(real_frac_val))
        gen_pct = tmp.sample(n_eval, sampler="ddim", ddim_steps=cfg["ddim_steps"])
        gen_frac = (gen_pct / 100.0).astype(np.float32)
        score, parts = _score_generated_surfaces(real_frac_val[:n_eval], gen_frac[:n_eval], m_new, tau_new, arb_weight=ARB_WEIGHT)
        trial.set_user_attr("arb", parts["arb"])
        trial.set_user_attr("swd", parts["swd"])
        return score

    ddpm_study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED + 1))
    ddpm_study.optimize(ddpm_objective, n_trials=int(HPO_TRIALS.get("ddpm", 8)), show_progress_bar=False)
    ddpm_best_trial = ddpm_study.best_trial
    ddpm_best_cfg = dict(ddpm_best_trial.params)
    ddpm_defaults.update(ddpm_best_cfg)
    _set_best(
        "iv_ddpm",
        ddpm_best_cfg,
        score=float(ddpm_best_trial.value),
        extra={"arb": float(ddpm_best_trial.user_attrs.get("arb", np.nan)), "swd": float(ddpm_best_trial.user_attrs.get("swd", np.nan)), "trial": int(ddpm_best_trial.number)},
    )
    print("Best DDPM params:", ddpm_best_cfg)

iv_ddpm = IVDDPM(
    tau_grid=tau_new,
    lo=lo_v,
    hi=hi_v,
    T=int(ddpm_defaults["T"]),
    enc_channels=int(ddpm_defaults["enc_channels"]),
    bot_channels=int(ddpm_defaults["bot_channels"]),
    emb_dim=int(ddpm_defaults["emb_dim"]),
    lambda_smile=float(ddpm_defaults["lambda_smile"]),
    lambda_ttm=float(ddpm_defaults["lambda_ttm"]),
    device=device,
)
iv_ddpm.fit(
    surf_train_n,
    surf_val_n=None,
    n_epochs=int(ddpm_defaults["n_epochs"]),
    batch_size=int(ddpm_defaults["batch_size"]),
    lr=float(ddpm_defaults["lr"]),
    print_every=100,
    patience=int(ddpm_defaults["patience"]),
)
iv_ddpm.save(IV_DDPM_CKPT)
set_all_seeds(SEED + 401)
ivs_ivddpm_raw = iv_ddpm.sample(N_GEN, sampler="ddim", ddim_steps=int(ddpm_defaults["ddim_steps"]))
print(
    f"DiffPen retrained and saved -> {IV_DDPM_CKPT} | "
    f"shape={ivs_ivddpm_raw.shape}, range=[{ivs_ivddpm_raw.min():.2f}%, {ivs_ivddpm_raw.max():.2f}%], "
    f"mean={ivs_ivddpm_raw.mean():.2f}%"
)


# VolGAN
set_all_seeds(SEED + 500)
surf_train_np = surf_train_n.detach().cpu().numpy()
surf_hpo_train_np = surf_hpo_train_n.detach().cpu().numpy()

volgan_defaults = {
    "noise_dim": 32,
    "hidden_dim": 128,
    "n_epochs": COMMON_FINAL_EPOCHS,
    "n_grad": 100,
    "lr_g": 1e-4,
    "lr_d": 1e-4,
    "batch_size": 32,
    "penalty_mode": "iv",
    "n_seeds": 10,
}

if USE_BEST_HP and "volgan" in HPO_BEST:
    volgan_best = dict(HPO_BEST["volgan"].get("params", {}))
    volgan_defaults.update(volgan_best)
    print("Using cached best VolGAN params:", volgan_best)
elif RUN_HPO and OPTUNA_AVAILABLE and len(surf_hpo_val_n) >= 16:
    print("Running Optuna for VolGAN...")

    def volgan_objective(trial):
        cfg = {
            "noise_dim": int(trial.suggest_categorical("noise_dim", [16, 24, 32, 48])),
            "hidden_dim": int(trial.suggest_categorical("hidden_dim", [64, 96, 128, 192])),
            "n_epochs": int(trial.suggest_int("n_epochs", 200, int(HPO_FAST_EPOCHS.get("volgan", 400)), step=100)),
            "n_grad": int(trial.suggest_categorical("n_grad", [30, 50, 80])),
            "lr_g": float(trial.suggest_float("lr_g", 5e-5, 4e-4, log=True)),
            "lr_d": float(trial.suggest_float("lr_d", 5e-5, 4e-4, log=True)),
            "batch_size": int(trial.suggest_categorical("batch_size", [16, 32, 64])),
            "n_seeds": int(trial.suggest_categorical("n_seeds", [5, 8, 10])),
        }
        tmp_path = f"data/treated/_tmp_volgan_hpo_trial_{trial.number}.pt"
        gen, _, _, _, _ = train_volgan(
            surf_train_n=surf_hpo_train_np,
            lo_v=lo_v,
            hi_v=hi_v,
            m_new=m_new,
            tau_new=tau_new,
            device=device,
            noise_dim=cfg["noise_dim"],
            hidden_dim=cfg["hidden_dim"],
            n_epochs=cfg["n_epochs"],
            n_grad=cfg["n_grad"],
            lr_g=cfg["lr_g"],
            lr_d=cfg["lr_d"],
            batch_size=cfg["batch_size"],
            penalty_mode="iv",
            print_every=10**9,
            save_path=tmp_path,
        )
        n_eval = min(512, len(real_frac_val))
        gen_pct = sample_volgan_diverse(
            gen=gen,
            surf_train_n=surf_hpo_train_np,
            N_GEN=n_eval,
            noise_dim=cfg["noise_dim"],
            lo_v=lo_v,
            hi_v=hi_v,
            device=device,
            n_seeds=cfg["n_seeds"],
        )
        gen_frac = (gen_pct / 100.0).astype(np.float32)
        score, parts = _score_generated_surfaces(real_frac_val[:n_eval], gen_frac[:n_eval], m_new, tau_new, arb_weight=ARB_WEIGHT)
        trial.set_user_attr("arb", parts["arb"])
        trial.set_user_attr("swd", parts["swd"])
        return score

    vol_study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED + 2))
    vol_study.optimize(volgan_objective, n_trials=int(HPO_TRIALS.get("volgan", 8)), show_progress_bar=False)
    vol_best_trial = vol_study.best_trial
    vol_best_cfg = dict(vol_best_trial.params)
    volgan_defaults.update(vol_best_cfg)
    _set_best(
        "volgan",
        vol_best_cfg,
        score=float(vol_best_trial.value),
        extra={"arb": float(vol_best_trial.user_attrs.get("arb", np.nan)), "swd": float(vol_best_trial.user_attrs.get("swd", np.nan)), "trial": int(vol_best_trial.number)},
    )
    print("Best VolGAN params:", vol_best_cfg)

volgan_ckpt = VOLGAN_CKPT
vg_gen, vg_disc, vg_gen_losses, vg_disc_losses, vg_arb_losses = train_volgan(
    surf_train_n=surf_train_np,
    lo_v=lo_v,
    hi_v=hi_v,
    m_new=m_new,
    tau_new=tau_new,
    device=device,
    noise_dim=int(volgan_defaults["noise_dim"]),
    hidden_dim=int(volgan_defaults["hidden_dim"]),
    n_epochs=int(volgan_defaults["n_epochs"]),
    n_grad=int(volgan_defaults["n_grad"]),
    lr_g=float(volgan_defaults["lr_g"]),
    lr_d=float(volgan_defaults["lr_d"]),
    batch_size=int(volgan_defaults["batch_size"]),
    penalty_mode=str(volgan_defaults["penalty_mode"]),
    print_every=100,
    save_path=volgan_ckpt,
)

set_all_seeds(SEED + 501)
vg_iv_raw = sample_volgan_diverse(
    gen=vg_gen,
    surf_train_n=surf_train_np,
    N_GEN=N_GEN,
    noise_dim=int(volgan_defaults["noise_dim"]),
    lo_v=lo_v,
    hi_v=hi_v,
    device=device,
    n_seeds=int(volgan_defaults["n_seeds"]),
)
ivs_volgan = vg_iv_raw
vg_available = True

print(
    f"VolGAN retrained and saved -> {volgan_ckpt} | "
    f"shape={vg_iv_raw.shape}, range=[{vg_iv_raw.min():.2f}%, {vg_iv_raw.max():.2f}%], "
    f"mean={vg_iv_raw.mean():.2f}%"
)
print(f"Hyperparameter cache updated at: {HPO_CACHE_PATH}")

In [ ]:
# VolGAN arbitrage reweighting 
if vg_iv_raw is None:
    print("VolGAN not available — skipping reweighting.")
    ivs_volgan = None
else:
    try:
        from src.other_models.VolGan import arbitrage_penalty_tensor, penalty_mutau_tensor, smallBS_tensor

        N_POOL    = len(vg_iv_raw)
        surfs_raw = vg_iv_raw                          # (N_POOL, 16, 32) in %
        surfs_t   = torch.tensor(surfs_raw / 100.0, dtype=torch.float32, device=device)
        # shape: (N_POOL, 16, 32) = (N_POOL, NT, NM)
        surfs_nm_nt = surfs_t.permute(0, 2, 1)        # (N_POOL, NM=32, NT=16)

        dtm     = tau_new * 365
        mP_t, mP_k, mPb_k = penalty_mutau_tensor(m_new, dtm, device)
        m_t     = torch.tensor(m_new,   dtype=torch.float32, device=device)
        tau_t   = torch.tensor(tau_new, dtype=torch.float32, device=device)

        BS = smallBS_tensor(
            m_t.view(1, -1, 1).expand(N_POOL, 32, 16),
            tau_t.view(1, 1, -1).expand(N_POOL, 32, 16),
            surfs_nm_nt, r=0
        )
        _, _, _, phi = arbitrage_penalty_tensor(
            BS,
            mP_t.unsqueeze(0).expand(N_POOL, -1, -1),
            mP_k.unsqueeze(0).expand(N_POOL, -1, -1),
            mPb_k.unsqueeze(0).expand(N_POOL, -1, -1),
        )
        phi_np  = phi.cpu().numpy()
        beta_rw = 3.0
        log_w   = -beta_rw * phi_np
        log_w  -= log_w.max()
        w       = np.exp(log_w); w /= w.sum()
        rw_rng = np.random.default_rng(SEED + 502)
        idx_rw = rw_rng.choice(N_POOL, size=5000, replace=True, p=w)
        ivs_volgan = surfs_raw[idx_rw]
        print(f"VolGAN reweighted: mean_phi_before={phi_np.mean():.5f}  "
              f"after={(w*phi_np).sum():.5f}  "
              f"pct_clean={(phi_np[idx_rw]<1e-4).mean()*100:.1f}%")
    except Exception as _rw_err:
        print(f"VolGAN reweighting skipped: {_rw_err}")
        ivs_volgan = vg_iv_raw    # fall back to raw


### latentSB / Cont–Vuletić path models 

If the path-model arrays already exist in the notebook kernel, they are reused. Otherwise the models are rerun directly from the folder layout:

- `src/other_models/cont_simulations.py` for the Cont–Vuletić PCA/AR(1)/WMC baseline;
- `src/volsurface_latentSB/sbjts_pca_vol_surface.py`;
- `src/volsurface_latentSB/sbjts_autoencoder_vol_surface.py`;
- `src/volsurface_latentSB/lightsb_vol_surface_github_torchcompat.py`

The returned `paths` arrays are converted from `(paths, time, moneyness, tenor)` to `(paths, time, tenor, moneyness)`, resized to `(16, 32)`, and inserted into `models_cmp` as generated surfaces while keeping the true path arrays separately for path diagnostics.

In [ ]:
# This cell reads CUBE_PATH = data/treated/vol_surface_spx_long.npz.
# It retrains/reruns path models on full sample and persists outputs
# (NPZ artifacts) to data/treated/ for reproducibility.

from pathlib import Path
import importlib
import sys
import warnings
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd

PROJECT_ROOT_CMP = Path.cwd().resolve()
SRC_DIR_CMP = PROJECT_ROOT_CMP / "src"
for _p in [PROJECT_ROOT_CMP, SRC_DIR_CMP]:
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))


def _import_any(module_names):
    last = None
    for name in module_names:
        try:
            return importlib.import_module(name)
        except Exception as exc:
            last = exc
    raise ImportError(f"Could not import any of {module_names}. Last error: {type(last).__name__}: {last}")



try:
    cont_mod = _import_any(["other_models.cont_simulations", "src.other_models.cont_simulations"])
except Exception as exc:
    cont_mod = None
   
try:
    sbjts_pca_mod = _import_any(["volsurface_latentSB.sbjts_pca_vol_surface", "src.volsurface_latentSB.sbjts_pca_vol_surface"])

except Exception as exc:
    sbjts_pca_mod = None


try:
    if sbjts_pca_mod is not None:
        try:
            latentSB_pkg = importlib.import_module("volsurface_latentSB")
            sys.modules.setdefault("volsurface_forecast", latentSB_pkg)
        except Exception:
            pass
        sys.modules.setdefault("volsurface_forecast.sbjts_pca_vol_surface", sbjts_pca_mod)
except Exception:
    pass

try:
    sbjts_ae_mod = _import_any(["volsurface_latentSB.sbjts_autoencoder_vol_surface", "src.volsurface_latentSB.sbjts_autoencoder_vol_surface"])
except Exception as exc:
    sbjts_ae_mod = None

try:
    lightsb_mod = _import_any(["volsurface_latentSB.lightsb_vol_surface_github_torchcompat", "src.volsurface_latentSB.lightsb_vol_surface_github_torchcompat"])
except Exception as exc:
    lightsb_mod = None

try:
    bridge_mod = _import_any(["volsurface_path_bridge", "src.volsurface_path_bridge"])
except Exception as exc:
    bridge_mod = None

#print("Project root:", PROJECT_ROOT_CMP)

# Shared run settings
FORCE_RERUN_MY_PATH_MODELS = bool(globals().get("FORCE_RERUN_MY_PATH_MODELS", True))
RUN_MY_PATH_MODELS_IF_MISSING = bool(globals().get("RUN_MY_PATH_MODELS_IF_MISSING", True))

MY_N_OUTPUT_PATHS = int(globals().get("MY_N_OUTPUT_PATHS", 50))
MY_PATH_LENGTH = int(globals().get("MY_PATH_LENGTH", 100))
MY_N_CANDIDATE_PATHS = int(globals().get("MY_N_CANDIDATE_PATHS", 512))
MY_CANDIDATE_BATCH_SIZE = int(globals().get("MY_CANDIDATE_BATCH_SIZE", 128))
MY_SEED = int(SEED + 600)
MY_BETA = float(globals().get("MY_BETA", 1.0e4))

# Model-specific defaults
MY_CONT_N_FACTORS = int(globals().get("MY_CONT_N_FACTORS", 4))
MY_SBJTS_PCA_N_FACTORS = int(globals().get("MY_SBJTS_PCA_N_FACTORS", 4))
MY_SBJTS_AE_LATENT_DIM = int(globals().get("MY_SBJTS_AE_LATENT_DIM", 8))
MY_SBJTS_AE_EPOCHS = int(globals().get("MY_SBJTS_AE_EPOCHS", COMMON_FINAL_EPOCHS))
MY_SBJTS_AE_BATCH_SIZE = int(globals().get("MY_SBJTS_AE_BATCH_SIZE", 64))
MY_LIGHTSB_N_FACTORS = int(globals().get("MY_LIGHTSB_N_FACTORS", 8))
MY_LIGHTSB_N_COMPONENTS = int(globals().get("MY_LIGHTSB_N_COMPONENTS", 32))
MY_LIGHTSB_EPOCHS = int(globals().get("MY_LIGHTSB_EPOCHS", COMMON_FINAL_EPOCHS))
MY_LIGHTSB_BATCH_SIZE = int(globals().get("MY_LIGHTSB_BATCH_SIZE", 64))
MY_LIGHTSB_EPSILON = float(globals().get("MY_LIGHTSB_EPSILON", 0.12))

print("\nPath-model settings:")
print(f"  FORCE_RERUN_MY_PATH_MODELS: {FORCE_RERUN_MY_PATH_MODELS}")
print(f"  n_output_paths: {MY_N_OUTPUT_PATHS}  path_length: {MY_PATH_LENGTH}  "
      f" so total surfaces per model: {MY_N_OUTPUT_PATHS * MY_PATH_LENGTH}")
print(f"  n_candidate_paths: {MY_N_CANDIDATE_PATHS}  batch: {MY_CANDIDATE_BATCH_SIZE}")
print(f"  cube input (SPX): {CUBE_PATH}")

def _to_fraction_iv(x):
    x = np.asarray(x, dtype=np.float32)
    return x / 100.0 if np.nanmedian(x) > 2.0 else x

def _resize_surfaces_t_m_fraction(surfaces_t_m_frac, target_shape=(16, 32)):
    x = np.asarray(surfaces_t_m_frac, dtype=np.float32)
    if x.ndim != 3:
        raise ValueError(f"Expected surfaces (N,T,M), got {x.shape}")
    if tuple(x.shape[-2:]) == tuple(target_shape):
        return x
    with torch.no_grad():
        t = torch.tensor(x, dtype=torch.float32).unsqueeze(1)
        y = F.interpolate(t, size=target_shape, mode="bilinear", align_corners=False)
    return y[:, 0].detach().cpu().numpy().astype(np.float32)

def _orient_paths_to_t_m(paths, m_grid_local=None, tenor_days_local=None, target_shape=(16, 32)):
    x = np.asarray(paths, dtype=np.float32)
    if x.ndim != 4:
        raise ValueError(f"paths must be 4D (P,L,*,*), got {x.shape}")
    a, b = x.shape[-2], x.shape[-1]
    n_m = len(m_grid_local) if m_grid_local is not None else None
    n_t = len(tenor_days_local) if tenor_days_local is not None else None
    if n_t is not None and n_m is not None:
        if (a, b) == (n_t, n_m):
            return x
        if (a, b) == (n_m, n_t):
            return np.swapaxes(x, -1, -2)
    if (a, b) == tuple(target_shape):
        return x
    if (a, b) == (target_shape[1], target_shape[0]):
        return np.swapaxes(x, -1, -2)
    return np.swapaxes(x, -1, -2) if a > b else x

def _result_to_paths_and_surfaces_fraction_16x32(result, label, target_shape=(16, 32)):
    if not hasattr(result, "paths"):
        raise ValueError(f"{label}: result has no .paths attribute")
    paths_raw = np.asarray(result.paths, dtype=np.float32)
    m_local = np.asarray(getattr(result, "m_grid", m_grid), dtype=float)
    tenor_local = np.asarray(getattr(result, "tenor_days", tenor_days), dtype=float)
    paths_t_m = _orient_paths_to_t_m(paths_raw, m_local, tenor_local, target_shape=target_shape)
    paths_frac = _to_fraction_iv(paths_t_m)
    P, L, T0, M0 = paths_frac.shape
    flat = paths_frac.reshape(P * L, T0, M0)
    flat16 = _resize_surfaces_t_m_fraction(flat, target_shape=target_shape)
    paths16 = flat16.reshape(P, L, target_shape[0], target_shape[1])
    return paths16, flat16

notebook8_model_results = globals().get("notebook8_model_results", {})
notebook8_path_models = globals().get("notebook8_path_models", {})
notebook8_models_cmp = globals().get("notebook8_models_cmp", {})
notebook8_model_sources = globals().get("notebook8_model_sources", {})

def _register_result(label, result, source):
    paths16, flat16 = _result_to_paths_and_surfaces_fraction_16x32(result, label, target_shape=(NT, NM))
    notebook8_model_results[label] = result
    notebook8_path_models[label] = paths16
    notebook8_models_cmp[label] = flat16
    notebook8_model_sources[label] = source
    print(f"  added {label:<30} paths={paths16.shape} surfaces={flat16.shape} source={source}")

def _save_result_if_possible(label, result, out_path):
    try:
        if hasattr(result, "candidate_path_diagnostics"):
            result.candidate_path_diagnostics.to_csv(
                out_path.with_name(out_path.stem + "_candidate_path_diagnostics.csv"),
                index=False,
            )
        if hasattr(result, "selected_path_diagnostics"):
            result.selected_path_diagnostics.to_csv(
                out_path.with_name(out_path.stem + "_selected_path_diagnostics.csv"),
                index=False,
            )
        if hasattr(result, "selected_surface_diagnostics"):
            result.selected_surface_diagnostics.to_csv(
                out_path.with_name(out_path.stem + "_selected_surface_diagnostics.csv"),
                index=False,
            )
    except Exception as _diag_err:
        print(f"  diagnostics save warning for {label}: {_diag_err}")

def _safe_run_and_save(label, fn, save_fn=None, out_path=None):
    print(f"\nRunning {label} ...")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        result = fn()
    if save_fn is not None and out_path is not None:
        save_fn(out_path, result)
        _save_result_if_possible(label, result, out_path)
        print(f"  saved artifact -> {out_path}")
    _register_result(label, result, "rerun on full sample + saved")

if "src.volsurface_latentFM.optuna_vae" in sys.modules:
    print("FM Optuna module detected in session; ignored for SB/Cont tuning.")
real_frac_cmp = _real_fraction_surfaces_16x32()
HPO_N_OUTPUT_PATHS = max(8, min(12, MY_N_OUTPUT_PATHS))
HPO_PATH_LENGTH = max(24, min(40, MY_PATH_LENGTH))
HPO_N_CANDIDATE_PATHS = max(64, min(128, MY_N_CANDIDATE_PATHS))
HPO_CANDIDATE_BATCH_SIZE = max(32, min(64, MY_CANDIDATE_BATCH_SIZE))

def _score_path_result(result, label="tmp"):
    _, flat16 = _result_to_paths_and_surfaces_fraction_16x32(result, label, target_shape=(NT, NM))
    n = min(len(flat16), len(real_frac_cmp))
    score, parts = _score_generated_surfaces(
        real_frac_cmp[:n],
        flat16[:n],
        m_common,
        tau_common,
        arb_weight=ARB_WEIGHT,
    )
    return score, parts

def _run_optuna_path_model(key, objective, n_trials):
    if USE_BEST_HP and key in HPO_BEST:
        params = HPO_BEST[key].get("params", {})
        print(f"Using cached best params for {key}: {params}")
        return dict(params)

    if not (RUN_HPO and OPTUNA_AVAILABLE):
        return None

    print(f"Running Optuna for {key} ({n_trials} trials)...")
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED + 11))
    study.optimize(objective, n_trials=int(n_trials), show_progress_bar=False)
    best = study.best_trial
    _set_best(
        key,
        dict(best.params),
        score=float(best.value),
        extra={
            "arb": float(best.user_attrs.get("arb", np.nan)),
            "swd": float(best.user_attrs.get("swd", np.nan)),
            "trial": int(best.number),
        },
    )
    print(f"Best params for {key}: {best.params} | score={best.value:.6f}")
    return dict(best.params)


print("\nRetraining/rerunning your path models (SPX cube):")

# Cont-Vuletic
if cont_mod is not None:
    cont_params = {
        "n_factors": int(MY_CONT_N_FACTORS),
        "beta": float(MY_BETA),
    }

    def _cont_hpo_objective(trial):
        cfg = cont_mod.SimulationConfig(
            input_path=CUBE_PATH,
            output_path=seed_path("_tmp_cont_hpo.npz"),
            n_output_paths=HPO_N_OUTPUT_PATHS,
            path_length=HPO_PATH_LENGTH,
            n_candidate_paths=HPO_N_CANDIDATE_PATHS,
            candidate_batch_size=HPO_CANDIDATE_BATCH_SIZE,
            n_factors=int(trial.suggest_int("n_factors", 3, 12)),
            seed=MY_SEED,
            beta=float(trial.suggest_float("beta", 1e3, 5e4, log=True)),
            require_zero_path=False,
            start_mode="last",
        )
        result = cont_mod.simulate_arbitrage_free_paths(cfg)
        score, parts = _score_path_result(result, "Cont HPO")
        trial.set_user_attr("arb", parts["arb"])
        trial.set_user_attr("swd", parts["swd"])
        return score

    best = _run_optuna_path_model("cont_vuletic", _cont_hpo_objective, HPO_TRIALS.get("cont", 8))
    if best is not None:
        cont_params.update(best)

    cont_out = seed_path("cont_vuletic_spx_full.npz")

    def _run_cont():
        cfg = cont_mod.SimulationConfig(
            input_path=CUBE_PATH,
            output_path=cont_out,
            n_output_paths=MY_N_OUTPUT_PATHS,
            path_length=MY_PATH_LENGTH,
            n_candidate_paths=MY_N_CANDIDATE_PATHS,
            candidate_batch_size=MY_CANDIDATE_BATCH_SIZE,
            n_factors=int(cont_params["n_factors"]),
            seed=MY_SEED,
            beta=float(cont_params["beta"]),
            require_zero_path=False,
            start_mode="last",
        )
        return cont_mod.simulate_arbitrage_free_paths(cfg)

    _safe_run_and_save("Cont–Vuletic PCA-AR1-WMC", _run_cont, cont_mod.save_result_npz, cont_out)

# SBJTS PCA
if sbjts_pca_mod is not None:
    sbjts_pca_params = {
        "n_factors": int(MY_SBJTS_PCA_N_FACTORS),
        "bandwidth": 1.0,
        "jump_quantile": 0.90,
        "beta": float(MY_BETA),
    }

    def _sbjts_pca_hpo_objective(trial):
        cfg = sbjts_pca_mod.SBJTSPCAConfig(
            input_path=CUBE_PATH,
            output_path=seed_path("_tmp_sbjts_pca_hpo.npz"),
            n_output_paths=HPO_N_OUTPUT_PATHS,
            path_length=HPO_PATH_LENGTH,
            n_candidate_paths=HPO_N_CANDIDATE_PATHS,
            candidate_batch_size=HPO_CANDIDATE_BATCH_SIZE,
            n_factors=int(trial.suggest_int("n_factors", 3, 12)),
            bandwidth=float(trial.suggest_float("bandwidth", 0.5, 2.0)),
            jump_quantile=float(trial.suggest_float("jump_quantile", 0.80, 0.98)),
            seed=MY_SEED + 101,
            beta=float(trial.suggest_float("beta", 1e3, 5e4, log=True)),
            require_zero_path=False,
            start_mode="last",
        )
        result = sbjts_pca_mod.simulate_sbjts_pca_paths(cfg)
        score, parts = _score_path_result(result, "SBJTS PCA HPO")
        trial.set_user_attr("arb", parts["arb"])
        trial.set_user_attr("swd", parts["swd"])
        return score

    best = _run_optuna_path_model("sbjts_pca", _sbjts_pca_hpo_objective, HPO_TRIALS.get("sbjts_pca", 8))
    if best is not None:
        sbjts_pca_params.update(best)

    sbjts_pca_out = seed_path("sbjts_pca_spx_full.npz")

    def _run_sbjts_pca():
        cfg = sbjts_pca_mod.SBJTSPCAConfig(
            input_path=CUBE_PATH,
            output_path=sbjts_pca_out,
            n_output_paths=MY_N_OUTPUT_PATHS,
            path_length=MY_PATH_LENGTH,
            n_candidate_paths=MY_N_CANDIDATE_PATHS,
            candidate_batch_size=MY_CANDIDATE_BATCH_SIZE,
            n_factors=int(sbjts_pca_params["n_factors"]),
            bandwidth=float(sbjts_pca_params["bandwidth"]),
            jump_quantile=float(sbjts_pca_params["jump_quantile"]),
            seed=MY_SEED + 101,
            beta=float(sbjts_pca_params["beta"]),
            require_zero_path=False,
            start_mode="last",
        )
        return sbjts_pca_mod.simulate_sbjts_pca_paths(cfg)

    _safe_run_and_save("SBJTS PCA", _run_sbjts_pca, sbjts_pca_mod.save_result_npz, sbjts_pca_out)

# SBJTS AE/VAE
if sbjts_ae_mod is not None:
    sbjts_ae_params = {
        "latent_dim": int(MY_SBJTS_AE_LATENT_DIM),
        "ae_epochs": int(MY_SBJTS_AE_EPOCHS),
        "ae_batch_size": int(MY_SBJTS_AE_BATCH_SIZE),
        "ae_lr": 1.0e-3,
        "beta": float(MY_BETA),
    }

    def _sbjts_ae_hpo_objective(trial):
        cfg = sbjts_ae_mod.SBJTSAEConfig(
            input_path=CUBE_PATH,
            output_path=seed_path("_tmp_sbjts_ae_hpo.npz"),
            autoencoder_checkpoint_path=seed_path("_tmp_sbjts_ae_hpo.pt"),
            n_output_paths=HPO_N_OUTPUT_PATHS,
            path_length=HPO_PATH_LENGTH,
            n_candidate_paths=HPO_N_CANDIDATE_PATHS,
            candidate_batch_size=HPO_CANDIDATE_BATCH_SIZE,
            latent_dim=int(trial.suggest_int("latent_dim", 4, 16)),
            ae_epochs=int(trial.suggest_int("ae_epochs", 20, int(HPO_FAST_EPOCHS.get("sbjts_ae", 40)), step=10)),
            ae_batch_size=int(trial.suggest_categorical("ae_batch_size", [32, 64, 96])),
            ae_lr=float(trial.suggest_float("ae_lr", 3e-4, 3e-3, log=True)),
            seed=MY_SEED + 202,
            beta=float(trial.suggest_float("beta", 1e3, 5e4, log=True)),
            require_zero_path=False,
            start_mode="last",
            load_autoencoder_if_exists=False,
            save_autoencoder=False,
            device=str(device),
        )
        result = sbjts_ae_mod.simulate_sbjts_autoencoder_paths(cfg)
        score, parts = _score_path_result(result, "SBJTS AE HPO")
        trial.set_user_attr("arb", parts["arb"])
        trial.set_user_attr("swd", parts["swd"])
        return score

    best = _run_optuna_path_model("sbjts_ae", _sbjts_ae_hpo_objective, HPO_TRIALS.get("sbjts_ae", 8))
    if best is not None:
        sbjts_ae_params.update(best)

    sbjts_ae_out = seed_path("sbjts_ae_spx_full.npz")

    def _run_sbjts_ae():
        cfg = sbjts_ae_mod.SBJTSAEConfig(
            input_path=CUBE_PATH,
            output_path=sbjts_ae_out,
            autoencoder_checkpoint_path=seed_path("sbjts_ae_spx_full.pt"),
            n_output_paths=MY_N_OUTPUT_PATHS,
            path_length=MY_PATH_LENGTH,
            n_candidate_paths=MY_N_CANDIDATE_PATHS,
            candidate_batch_size=MY_CANDIDATE_BATCH_SIZE,
            latent_dim=int(sbjts_ae_params["latent_dim"]),
            ae_epochs=int(sbjts_ae_params["ae_epochs"]),
            ae_batch_size=int(sbjts_ae_params["ae_batch_size"]),
            ae_lr=float(sbjts_ae_params["ae_lr"]),
            seed=MY_SEED + 202,
            beta=float(sbjts_ae_params["beta"]),
            require_zero_path=False,
            start_mode="last",
            load_autoencoder_if_exists=False,
            save_autoencoder=True,
            device=str(device),
        )
        return sbjts_ae_mod.simulate_sbjts_autoencoder_paths(cfg)

    _safe_run_and_save("SBJTS AE/VAE", _run_sbjts_ae, sbjts_ae_mod.save_result_npz, sbjts_ae_out)

# LightSB
if lightsb_mod is not None:
    lightsb_params = {
        "n_factors": int(MY_LIGHTSB_N_FACTORS),
        "n_components": int(MY_LIGHTSB_N_COMPONENTS),
        "epsilon": float(MY_LIGHTSB_EPSILON),
        "epochs": int(MY_LIGHTSB_EPOCHS),
        "batch_size": int(MY_LIGHTSB_BATCH_SIZE),
        "lr": 2.0e-3,
        "beta": float(MY_BETA),
    }

    def _lightsb_hpo_objective(trial):
        cfg = lightsb_mod.LightSBVolConfig(
            input_path=CUBE_PATH,
            output_path=seed_path("_tmp_lightsb_hpo.npz"),
            n_output_paths=HPO_N_OUTPUT_PATHS,
            path_length=HPO_PATH_LENGTH,
            n_candidate_paths=HPO_N_CANDIDATE_PATHS,
            candidate_batch_size=HPO_CANDIDATE_BATCH_SIZE,
            n_factors=int(trial.suggest_int("n_factors", 4, 12)),
            n_components=int(trial.suggest_categorical("n_components", [16, 24, 32, 48, 64])),
            epsilon=float(trial.suggest_float("epsilon", 0.05, 0.25)),
            epochs=int(trial.suggest_int("epochs", 80, int(HPO_FAST_EPOCHS.get("lightsb", 200)), step=40)),
            batch_size=int(trial.suggest_categorical("batch_size", [32, 64, 128])),
            lr=float(trial.suggest_float("lr", 7e-4, 4e-3, log=True)),
            seed=MY_SEED + 303,
            beta=float(trial.suggest_float("beta", 1e3, 5e4, log=True)),
            require_zero_path=False,
            start_mode="last",
            device=str(device),
        )
        result = lightsb_mod.simulate_lightsb_paths(cfg)
        score, parts = _score_path_result(result, "LightSB HPO")
        trial.set_user_attr("arb", parts["arb"])
        trial.set_user_attr("swd", parts["swd"])
        return score

    best = _run_optuna_path_model("lightsb", _lightsb_hpo_objective, HPO_TRIALS.get("lightsb", 8))
    if best is not None:
        lightsb_params.update(best)

    lightsb_out = seed_path("lightsb_spx_full.npz")

    def _run_lightsb():
        cfg = lightsb_mod.LightSBVolConfig(
            input_path=CUBE_PATH,
            output_path=lightsb_out,
            n_output_paths=MY_N_OUTPUT_PATHS,
            path_length=MY_PATH_LENGTH,
            n_candidate_paths=MY_N_CANDIDATE_PATHS,
            candidate_batch_size=MY_CANDIDATE_BATCH_SIZE,
            n_factors=int(lightsb_params["n_factors"]),
            n_components=int(lightsb_params["n_components"]),
            epsilon=float(lightsb_params["epsilon"]),
            epochs=int(lightsb_params["epochs"]),
            batch_size=int(lightsb_params["batch_size"]),
            lr=float(lightsb_params["lr"]),
            seed=MY_SEED + 303,
            beta=float(lightsb_params["beta"]),
            require_zero_path=False,
            start_mode="last",
            device=str(device),
        )
        return lightsb_mod.simulate_lightsb_paths(cfg)

    _safe_run_and_save("LightSB PCA level", _run_lightsb, lightsb_mod.save_result_npz, lightsb_out)

# Optional AR-residual bridge
if bridge_mod is not None:
    ar_fn = None
    for _fn_name in [
        "simulate_lightsb_ar_residual_paths",
        "simulate_lightsb_ar_residual",
        "run_lightsb_ar_residual_paths",
    ]:
        if hasattr(bridge_mod, _fn_name):
            ar_fn = getattr(bridge_mod, _fn_name)
            break
    if ar_fn is not None:
        def _run_ar_residual():
            try:
                return ar_fn(
                    input_path=CUBE_PATH,
                    n_output_paths=MY_N_OUTPUT_PATHS,
                    path_length=MY_PATH_LENGTH,
                    n_candidate_paths=MY_N_CANDIDATE_PATHS,
                    candidate_batch_size=MY_CANDIDATE_BATCH_SIZE,
                    seed=MY_SEED + 404,
                    device=str(device),
                    save=False,
                )
            except TypeError:
                return ar_fn({
                    "input_path": CUBE_PATH,
                    "n_output_paths": MY_N_OUTPUT_PATHS,
                    "path_length": MY_PATH_LENGTH,
                    "n_candidate_paths": MY_N_CANDIDATE_PATHS,
                    "candidate_batch_size": MY_CANDIDATE_BATCH_SIZE,
                    "seed": MY_SEED + 404,
                    "device": str(device),
                    "save": False,
                })

        print("\nRunning LightSB PCA AR-residual ...")
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ar_result = _run_ar_residual()
        _register_result("LightSB PCA AR-residual", ar_result, "rerun on full sample")
        try:
            _ar_paths = np.asarray(ar_result.paths, dtype=np.float32)
            np.savez_compressed(
                str(seed_path("lightsb_spx_ar_residual_full.npz")),
                paths=_ar_paths,
                m_grid=np.asarray(getattr(ar_result, "m_grid", m_grid)),
                tenor_days=np.asarray(getattr(ar_result, "tenor_days", tenor_days)),
            )
            print("  saved artifact -> data/treated/lightsb_spx_ar_residual_full.npz")
        except Exception as _ar_save_err:
            print(f"  AR-residual save warning: {_ar_save_err}")
    else:
        print("\nLightSB PCA AR-residual skipped: no callable AR-residual function found in volsurface_path_bridge.")

print("\nYour in-memory models available for comparison:")
for name, arr in notebook8_models_cmp.items():
    print(f"  {name:<30} surfaces={arr.shape} paths={notebook8_path_models[name].shape}")

print(f"\nHyperparameter cache updated at: {HPO_CACHE_PATH}")

In [ ]:
# Unified comparison 
import os
import seaborn as sns
from scipy.stats import norm
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np


def _ensure_fraction_16x32(arr, name="array"):
    """Accept % or fraction, resize to (N,NT,NM), return fraction."""
    x = np.asarray(arr, dtype=np.float32)
    if x.ndim == 4 and x.shape[1] == 1:
        x = x[:, 0]
    if x.ndim != 3:
        raise ValueError(f"{name}: expected (N,T,M) or (N,1,T,M), got {x.shape}")
    x = _to_fraction_iv(x) if "_to_fraction_iv" in globals() else (x / 100.0 if np.nanmedian(x) > 2.0 else x)
    if "_resize_surfaces_t_m_fraction" in globals():
        x = _resize_surfaces_t_m_fraction(x, target_shape=(NT, NM))
    return x

real_np_cmp = _ensure_fraction_16x32(real_iv[:split, 0].detach().cpu().numpy(), "real_np_cmp")
l_fm_cmp    = _ensure_fraction_16x32(ivs_ldm_fm_trigo, "L-FM")
vd_np       = _ensure_fraction_16x32(ivs_ivddpm_raw, "Diffusion")
vg_np       = _ensure_fraction_16x32(vg_iv_raw, "VolGAN raw")
vgw_np      = _ensure_fraction_16x32(ivs_volgan, "VolGAN reweighted")

models_cmp = {
    'Data (target)' : real_np_cmp,
    'L-FM linear'    : _ensure_fraction_16x32(ivs_ldm_fm_linear_raw, 'L-FM linear'),
    'L-FM quadratic' : _ensure_fraction_16x32(ivs_ldm_fm_quad_raw, 'L-FM quadratic'),
    'L-FM VP'        : _ensure_fraction_16x32(ivs_ldm_fm_vp_raw, 'L-FM VP'),
    'L-FM trigo'     : _ensure_fraction_16x32(ivs_ldm_fm_trigo, 'L-FM trigo'),
    'Latent DDPM'    : _ensure_fraction_16x32(ivs_latent_ddpm_raw, 'Latent DDPM'),
    'Diffusion'     : vd_np,
    'VolGAN'        : (_ensure_fraction_16x32(ivs_volgan, 'VolGAN') if ivs_volgan is not None else None),
    'VolGAN (raw)'  : (_ensure_fraction_16x32(vg_iv_raw, 'VolGAN (raw)') if vg_iv_raw is not None else None),
}

if "notebook8_models_cmp" in globals() and len(notebook8_models_cmp):
    for name, arr in notebook8_models_cmp.items():
        models_cmp[name] = _ensure_fraction_16x32(arr, name)

clr_cmp = {
    'Data (target)'              : '#222222',
    'L-FM linear'                : '#E07B00',
    'L-FM quadratic'             : '#FF6B6B',
    'L-FM VP'                    : '#C44E52',
    'L-FM trigo'                 : '#8B0000',
    'L-FM'                       : '#E07B00',   # legacy alias
    'Latent DDPM'                : '#4C78A8',
    'Diffusion'                  : '#1F77B4',
    'VolGAN'                     : '#B41F80',
    'VolGAN (raw)'               : '#9467BD',
    'Cont–Vuletic PCA-AR1-WMC'   : '#2CA02C',
    'SBJTS PCA'                  : '#D62728',
    'SBJTS AE/VAE'               : '#8C564B',
    'LightSB PCA level'          : '#17BECF',
    'LightSB PCA AR-residual'    : '#FF9896',
}
ls_cmp = {
    'Data (target)'              : '-',
    'L-FM linear'                : '--',
    'L-FM quadratic'             : '-.',
    'L-FM VP'                    : ':',
    'L-FM trigo'                 : 'dashed',
    'L-FM'                       : '--',   # legacy alias
    'Latent DDPM'                : (0, (5, 2)),
    'Diffusion'                  : '-.',
    'VolGAN'                     : 'dotted',
    'VolGAN (raw)'               : ':',
    'Cont–Vuletic PCA-AR1-WMC'   : '-',
    'SBJTS PCA'                  : '--',
    'SBJTS AE/VAE'               : '-.',
    'LightSB PCA level'          : ':',
    'LightSB PCA AR-residual'    : 'dashed',
}

# Dynamic fallback style for any extra models.
_palette = ['#4C78A8','#F58518','#54A24B','#E45756','#72B7B2','#B279A2','#FF9DA6','#9D755D','#BAB0AC']
_styles = ['-', '--', '-.', ':', 'dashed', 'dashdot']
for i, name in enumerate(models_cmp):
    clr_cmp.setdefault(name, _palette[i % len(_palette)])
    ls_cmp.setdefault(name, _styles[i % len(_styles)])

print("Models included in comparison:")
for name, arr in models_cmp.items():
    source = "retrained in current notebook"
    if "notebook8_model_sources" in globals() and name in notebook8_model_sources:
        source = notebook8_model_sources[name]
    print(f"  {name:<30} shape={arr.shape} median={np.nanmedian(arr):.4f} source={source}")

os.makedirs('figs', exist_ok=True)
mpl.rcParams.update(PAPER_RC)
sns.set_theme(context="paper", style="ticks", rc=PAPER_RC)
LW = 2.4

# Fig 1: IV Density 
fig, ax = plt.subplots(figsize=(8.2, 5.0))
for name, arr in models_cmp.items():
    sns.kdeplot(arr.reshape(-1), ax=ax, label=name,
                color=clr_cmp[name], linewidth=LW, linestyle=ls_cmp[name])
ax.set_xlim(real_np_cmp.min() * 0.85, max(0.62, np.nanpercentile(real_np_cmp, 99.5) * 1.25))
ax.set_xlabel('Implied Volatility')
ax.set_ylabel('Density')
ax.set_title('Implied Volatility Distribution')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('figs/fig_density_all.pdf', bbox_inches='tight')
plt.show()

# Summary statistics
print(f"\n{'Model':<30} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'Skew':>8}")
print('-' * 82)
for name, arr in models_cmp.items():
    x = arr.reshape(-1)
    skew = ((x - x.mean())**3).mean() / max(x.std()**3, 1e-12)
    print(f"{name:<30} {x.mean():8.4f} {x.std():8.4f} {x.min():8.4f} {x.max():8.4f} {skew:8.4f}")

# Fig 2: Average Volatility Smile
t_indices = [2, NT // 2, NT - 3]
t_labels = [
    rf'Short  $\tau={tau_new[t_indices[0]]:.2f}$ yr',
    rf'Medium $\tau={tau_new[t_indices[1]]:.2f}$ yr',
    rf'Long   $\tau={tau_new[t_indices[2]]:.2f}$ yr',
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5.0), sharey=True)
for k, (ax, t_idx, t_lbl) in enumerate(zip(axes, t_indices, t_labels)):
    for name, arr in models_cmp.items():
        ax.plot(m_new, arr[:, t_idx, :].mean(0),
                label=name, color=clr_cmp[name], linewidth=LW, linestyle=ls_cmp[name])
    ax.set_title(t_lbl)
    ax.set_xlabel('Moneyness')
    if k == 0:
        ax.set_ylabel('Implied Volatility')
    ax.legend(loc='upper right', fontsize=PAPER_FONT_SIZE)
fig.suptitle('Average Volatility Smile', fontsize=PAPER_FONT_SIZE, y=0.92)
plt.tight_layout()
plt.savefig('figs/fig_smile_all.pdf', bbox_inches='tight')
plt.show()

# Fig 3: Average Volatility Skew 
fig, axes = plt.subplots(1, 3, figsize=(16, 5.0), sharey=True)
for k, (ax, t_idx, t_lbl) in enumerate(zip(axes, t_indices, t_labels)):
    for name, arr in models_cmp.items():
        skews = np.gradient(arr[:, t_idx, :], m_new, axis=1)
        ax.plot(m_new, skews.mean(0), label=name,
                color=clr_cmp[name], linewidth=LW, linestyle=ls_cmp[name])
    ax.axhline(0, color='0.55', linestyle=':', linewidth=1.4)
    ax.set_title(t_lbl)
    ax.set_xlabel('Moneyness')
    if k == 0:
        ax.set_ylabel(r'$\partial\,\sigma_{\mathrm{IV}}\,/\,\partial K$')
    ax.legend(loc='upper left', fontsize=PAPER_FONT_SIZE)
fig.suptitle('Average Volatility Skew', fontsize=PAPER_FONT_SIZE, y=0.92)
plt.tight_layout()
plt.savefig('figs/fig_skew_all.pdf', bbox_inches='tight')
plt.show()

mpl.rcParams.update(PAPER_RC)

# Arbitrage penalties dynamically for all models
if "batch_phi" not in globals():
    raise RuntimeError("Run the pipeline-control/helper cell before this comparison cell.")
results = {name: batch_phi(arr, m_new, tau_new) for name, arr in models_cmp.items()}
COLORS = {name: clr_cmp[name] for name in models_cmp}
keys = ['phi', 'calendar', 'call', 'butterfly']
labels = [r'$\Phi$ total', r'$p_1$ calendar', r'$p_2$ monotonicity', r'$p_3$ butterfly']
model_names = list(results.keys())

# Summary table
print("\nMean no-arbitrage penalties")
print(f"{'Penalty':<18}" + "".join(f"{name[:18]:>20}" for name in model_names))
print('-' * (18 + 20 * len(model_names)))
for key, lbl in zip(keys, ['phi (total)', 'p1 calendar', 'p2 moneyness', 'p3 butterfly']):
    row = ''.join(f"{results[n][key].mean():20.5f}" for n in model_names)
    print(f"{lbl:<18}{row}")



In [ ]:
import hashlib
import json
import platform
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

RUN_DIR = SEED_ROOT
RUN_DIR.mkdir(parents=True, exist_ok=True)


def _safe_key(name):
    return "".join(ch.lower() if ch.isalnum() else "_" for ch in name).strip("_")


def _jsonable(x):
    if isinstance(x, dict):
        return {str(k): _jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_jsonable(v) for v in x]
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (np.integer, np.floating)):
        return x.item()
    if isinstance(x, Path):
        return str(x)
    return x


def _sha256(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while True:
            chunk = fh.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


key_map = {_safe_key(name): name for name in models_cmp}
npz_payload = {
    "m_grid": np.asarray(m_new, dtype=np.float64),
    "tenor_days": np.asarray(tau_new, dtype=np.float64) * 365.0,
}
if asof_dates is not None:
    npz_payload["asof_dates"] = np.asarray(asof_dates)
for key, name in key_map.items():
    npz_payload[key] = np.asarray(models_cmp[name], dtype=np.float32)

GENERATED_BUNDLE = seed_path("generated_surfaces_all_models.npz")
np.savez_compressed(GENERATED_BUNDLE, **npz_payload)

checkpoint_candidates = {
    "AE": AE_CKPT,
    **{f"FM_{k}": v for k, v in FM_CKPTS.items()},
    "Latent_DDPM": LATENT_DDPM_CKPT,
    "IV_DDPM": IV_DDPM_CKPT,
    "VolGAN": VOLGAN_CKPT,
    "Cont_Vuletic": str(seed_path("cont_vuletic_spx_full.npz")),
    "SBJTS_PCA": str(seed_path("sbjts_pca_spx_full.npz")),
    "SBJTS_AE_output": str(seed_path("sbjts_ae_spx_full.npz")),
    "SBJTS_AE_checkpoint": str(seed_path("sbjts_ae_spx_full.pt")),
    "LightSB": str(seed_path("lightsb_spx_full.npz")),
    "LightSB_AR_residual": str(seed_path("lightsb_spx_ar_residual_full.npz")),
}
checkpoint_status = {
    name: {"path": str(path), "exists": Path(path).exists()}
    for name, path in checkpoint_candidates.items()
}

try:
    git_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True,
        stderr=subprocess.DEVNULL,
    ).strip()
except Exception:
    git_commit = None

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "training_mode": "full-sample in-sample distribution comparison",
    "seed": SEED,
    "seed_set": list(SEEDS),
    "source_cube": str(CUBE_PATH),
    "source_cube_sha256": _sha256(CUBE_PATH),
    "source_cube_shape": list(cube_raw.shape),
    "m_grid": np.asarray(m_new).tolist(),
    "tenor_days": (np.asarray(tau_new) * 365.0).tolist(),
    "model_key_map": key_map,
    "model_shapes": {name: list(np.asarray(arr).shape) for name, arr in models_cmp.items()},
    "checkpoints": checkpoint_status,
    "hyperparameter_cache": str(HPO_CACHE_PATH),
    "hyperparameters": _jsonable({
        "AE": {"model": ae_params, "fit": ae_fit_cfg},
        "FM": fm_defaults,
        "Latent_DDPM": LATENT_DDPM_CFG,
        "IV_DDPM": ddpm_defaults,
        "VolGAN": volgan_defaults,
    }),
    "software": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "device": str(device),
    },
    "git_commit": git_commit,
    "generated_bundle": str(GENERATED_BUNDLE),
}

MANIFEST_PATH = seed_path("run_manifest.json")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True))

print("Saved reproducibility bundle:")
print(" ", GENERATED_BUNDLE)
print(" ", MANIFEST_PATH)
print("Existing checkpoints:")
for name, info in checkpoint_status.items():
    print(f"  {name:<24} {info['exists']}  {info['path']}")


### Optional path diagnostics for the true path generators




In [ ]:
import numpy as np
import pandas as pd

if "notebook8_path_models" not in globals() or not len(notebook8_path_models):
    print("No Notebook-8 path models available. Run the loader cell first.")
else:
    rows = []
    for name, paths in notebook8_path_models.items():
        x = np.asarray(paths, dtype=np.float32)
        if x.ndim != 4 or x.shape[1] < 2:
            continue
        # Δlog-IV realism: pathwise innovations, safe floor.
        dlog = np.diff(np.log(np.clip(x, 1e-6, None)), axis=1)
        step_l2 = np.sqrt(np.mean(dlog.reshape(dlog.shape[0], dlog.shape[1], -1) ** 2, axis=2))
        atm_t = min(NT // 2, x.shape[2] - 1)
        atm_m = int(np.argmin(np.abs(m_new - 1.0))) if np.nanmin(m_new) <= 1.0 <= np.nanmax(m_new) else x.shape[3] // 2
        atm_series = x[:, :, atm_t, atm_m]
        # Lag-1 autocorr averaged over paths, ignoring degenerate paths.
        acfs = []
        for p in range(atm_series.shape[0]):
            s = atm_series[p]
            if np.std(s[:-1]) > 1e-10 and np.std(s[1:]) > 1e-10:
                acfs.append(np.corrcoef(s[:-1], s[1:])[0, 1])
        rows.append({
            "model": name,
            "paths_shape": tuple(x.shape),
            "mean_iv": float(np.nanmean(x)),
            "std_iv": float(np.nanstd(x)),
            "mean_abs_dlog": float(np.nanmean(np.abs(dlog))),
            "mean_step_l2": float(np.nanmean(step_l2)),
            "p95_step_l2": float(np.nanpercentile(step_l2, 95)),
            "atm_lag1_autocorr_mean": float(np.nanmean(acfs)) if acfs else np.nan,
            "source": notebook8_model_sources.get(name, ""),
        })
    notebook8_path_diagnostics_table = pd.DataFrame(rows).sort_values("model")
    display(notebook8_path_diagnostics_table)


In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np

os.makedirs('figs', exist_ok=True)

mpl.rcParams.update(PAPER_RC)

means = {name: arr.mean(axis=0) * 100 for name, arr in models_cmp.items()}

all_vals = np.concatenate([v.ravel() for v in means.values()])
vmin = np.percentile(all_vals, 1)
vmax = np.percentile(all_vals, 99)

M, T = np.meshgrid(m_new, tau_new)   # M, T : (Ntau=16, Nm=32)

norm_surf = mcolors.Normalize(vmin=vmin, vmax=vmax)
cmap      = cm.RdYlGn_r

n      = len(means)
ELEV   = 25
AZIM   = -55

fig = plt.figure(figsize=(4.2 * n, 4.5))

for k, (name, surf) in enumerate(means.items()):
    ax = fig.add_subplot(1, n, k + 1, projection='3d')
    ax.plot_surface(
        M, T, surf,
        facecolors  = cmap(norm_surf(surf)),
        linewidth   = 0,
        antialiased = True,
        shade       = True,
        alpha       = 0.95,
    )
    ax.set_title(name, pad=6)
    ax.set_xlabel('Moneyness', labelpad=4)
    ax.set_ylabel('Tenor (yr)', labelpad=4)
    if k == 0:
        ax.set_zlabel('IV (%)', labelpad=4)
    ax.set_zlim(vmin, vmax)
    ax.view_init(elev=ELEV, azim=AZIM)
    ax.tick_params(axis='both', pad=1)
    ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(4))
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(4))
    ax.zaxis.set_major_locator(mpl.ticker.MaxNLocator(4))

sm = cm.ScalarMappable(cmap=cmap, norm=norm_surf)
sm.set_array([])
cbar = fig.colorbar(sm, ax=fig.axes, shrink=0.55, pad=0.04, aspect=25)
cbar.set_label('Mean IV (%)', fontsize=PAPER_FONT_SIZE)

fig.suptitle('Mean Implied Volatility Surface', fontsize=PAPER_FONT_SIZE, y=1.01)
plt.savefig('figs/fig_mean_surfaces_3d.pdf', bbox_inches='tight')
plt.show()

mpl.rcParams.update(PAPER_RC)

In [ ]:
# Q5, Q95 & Q99 IV Surfaces (3D) per Model 
import os
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np

os.makedirs('figs', exist_ok=True)

mpl.rcParams.update(PAPER_RC)

QUANTILES  = [5, 95, 99]
ELEV, AZIM = 25, -55
cmap       = cm.RdYlGn_r
M, T       = np.meshgrid(m_new, tau_new)   # (16, 32)
n_models   = len(models_cmp)

for q in QUANTILES:
    q_surfs = {name: np.percentile(arr * 100, q, axis=0)
               for name, arr in models_cmp.items()}

    all_vals  = np.concatenate([v.ravel() for v in q_surfs.values()])
    vmin      = np.percentile(all_vals, 1)
    vmax      = np.percentile(all_vals, 99)
    norm_surf = mcolors.Normalize(vmin=vmin, vmax=vmax)

    fig = plt.figure(figsize=(4.2 * n_models, 4.5))

    for k, (name, surf) in enumerate(q_surfs.items()):
        ax = fig.add_subplot(1, n_models, k + 1, projection='3d')
        ax.plot_surface(
            M, T, surf,
            facecolors  = cmap(norm_surf(surf)),
            linewidth   = 0,
            antialiased = True,
            shade       = True,
            alpha       = 0.95,
        )
        ax.set_title(name, pad=1, y=1.02)
        ax.set_xlabel('Moneyness', labelpad=0.7, fontsize=PAPER_FONT_SIZE)
        ax.set_ylabel('Tenor (yr)', labelpad=0.7, fontsize=PAPER_FONT_SIZE)
        ax.set_zlim(vmin, vmax)
        ax.view_init(elev=ELEV, azim=AZIM)
        ax.tick_params(axis='both', pad=1)
        ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(4))
        ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(4))
        ax.zaxis.set_major_locator(mpl.ticker.MaxNLocator(4))

    sm = cm.ScalarMappable(cmap=cmap, norm=norm_surf)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=fig.axes, shrink=0.55, pad=0.04, aspect=25)
    cbar.set_label('IV (%)', fontsize=PAPER_FONT_SIZE)

    fig.suptitle(f'{q}th-Percentile Implied Volatility Surface', fontsize=PAPER_FONT_SIZE, y=0.83)
    plt.savefig(f'figs/fig_q{q}_surfaces_3d.pdf', bbox_inches='tight')
    plt.show()

mpl.rcParams.update(PAPER_RC)

In [ ]:
# Wasserstein Distance Comparison + Tail/Regime Objective 
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import wasserstein_distance

os.makedirs('figs', exist_ok=True)

def sliced_wasserstein(X, Y, n_proj=300, seed=42):
    """Sliced W distance between (N,D) and (M,D) sample arrays."""
    rng = np.random.default_rng(seed)
    D, w = X.shape[1], 0.0
    for _ in range(n_proj):
        v  = rng.standard_normal(D)
        v /= np.linalg.norm(v)
        px = np.sort(X @ v)
        py = np.sort(Y @ v)
        N  = max(len(px), len(py))
        px = np.interp(np.linspace(0, 1, N), np.linspace(0, 1, len(px)), px)
        py = np.interp(np.linspace(0, 1, N), np.linspace(0, 1, len(py)), py)
        w += np.mean(np.abs(px - py))
    return w / n_proj

def regime_masks(means, thresholds):
    t1, t2, t3 = thresholds
    return [
        means < t1,
        (means >= t1) & (means < t2),
        (means >= t2) & (means < t3),
        means >= t3,
    ]

def _surface_level(arr):
    """Average IV level per surface, used to define volatility regimes."""
    arr = np.asarray(arr, dtype=float)
    return arr.mean(axis=(1, 2))

def _regime_proportions(arr, thresholds):
    """Empirical frequency of low/mid/high/extreme regimes."""
    masks = regime_masks(_surface_level(arr), thresholds)
    return np.array([float(mask.mean()) for mask in masks], dtype=float)

def _penalise_missing_metric(metric_dict, factor=1.25):
    """
    Replace NaN regime-W1 values by a finite penalty.
    This avoids rewarding a model that simply generates no surfaces in a regime.
    """
    finite = np.array([v for v in metric_dict.values() if np.isfinite(v)], dtype=float)
    if len(finite) == 0:
        fallback = 1.0
    else:
        worst = float(np.max(finite))
        fallback = factor * worst if worst > 0 else factor
    return {k: (float(v) if np.isfinite(v) else fallback) for k, v in metric_dict.items()}

def _minmax_normalise(metric_dict):
    """Normalise one metric across models. Lower is still better."""
    vals = np.array([metric_dict[n] for n in model_names_w], dtype=float)
    lo, hi = float(np.nanmin(vals)), float(np.nanmax(vals))
    if not np.isfinite(lo) or not np.isfinite(hi) or abs(hi - lo) < 1e-14:
        return {n: 0.0 for n in model_names_w}
    return {n: float((metric_dict[n] - lo) / (hi - lo)) for n in model_names_w}

# Data 
gen_models_w  = {k: np.asarray(v, dtype=float) for k, v in models_cmp.items()
                 if k != 'Data (target)'}
real_arr      = np.asarray(real_np_cmp, dtype=float)      # (N_real, 16, 32), fractions
real_flat     = real_arr.ravel()
X_real        = real_arr.reshape(len(real_arr), -1)       # (N_real, 512)
model_names_w = list(gen_models_w.keys())

QUANTILE_LEVELS  = [5, 25, 50, 75, 95, 99]
REGIME_LABELS    = ['Low\n(<Q25)', 'Mid\n(Q25–Q75)', 'High\n(Q75–Q95)', 'Extreme\n(>Q95)']
REGIME_PRINT     = [lbl.replace('\n', ' ') for lbl in REGIME_LABELS]
real_means       = _surface_level(real_arr)
regime_thresholds = np.percentile(real_means, [25, 75, 95])
real_regime_props = _regime_proportions(real_arr, regime_thresholds)

# Global W1 (flattened IV distribution) 
print("Global W1 …")
w1_global = {n: wasserstein_distance(real_flat, arr.ravel())
             for n, arr in gen_models_w.items()}

# Sliced Wasserstein on surface vectors 
print("Sliced Wasserstein (300 projections) …")
swd = {}
for n, arr in gen_models_w.items():
    swd[n] = sliced_wasserstein(X_real, arr.reshape(len(arr), -1))
    print(f"  {n:<28} {swd[n]:.6f}")

# W1 on pointwise quantile surfaces 
print("W1 per quantile surface …")
w1_q = {
    q: {
        n: wasserstein_distance(
            np.percentile(real_arr * 100, q, axis=0).ravel(),
            np.percentile(arr      * 100, q, axis=0).ravel(),
        )
        for n, arr in gen_models_w.items()
    }
    for q in QUANTILE_LEVELS
}

# W1 within volatility-regime sub-distributions
real_masks = regime_masks(real_means, regime_thresholds)
w1_regime_raw = {}
coverage_l1   = {}
regime_props  = {}

for n, arr in gen_models_w.items():
    regime_props[n] = _regime_proportions(arr, regime_thresholds)
    coverage_l1[n]  = float(np.abs(regime_props[n] - real_regime_props).sum())

w1_regime = {}
for lbl, rmask in zip(REGIME_LABELS, real_masks):
    w1_regime_raw[lbl] = {}
    w1_regime[lbl] = {}
    regime_idx = REGIME_LABELS.index(lbl)
    r_sub = real_arr[rmask].ravel()
    for n, arr in gen_models_w.items():
        mm = regime_masks(_surface_level(arr), regime_thresholds)
        m_sub = arr[mm[regime_idx]].ravel()
        raw_val = (wasserstein_distance(r_sub, m_sub)
                   if len(r_sub) > 0 and len(m_sub) > 0 else np.nan)
        w1_regime_raw[lbl][n] = raw_val

    w1_regime[lbl] = _penalise_missing_metric(w1_regime_raw[lbl], factor=1.25)

# Integrated tail/regime-aware objective score
TAIL_SCORE_WEIGHTS = {
    'W1 global flat' : 0.10,
    'Sliced W'       : 0.10,
    'Low regime W1'  : 0.05,
    'Mid regime W1'  : 0.05,
    'High regime W1' : 0.20,
    'Extreme W1'     : 0.35,
    'Coverage L1'    : 0.15,
}

norm_w1_global = _minmax_normalise(w1_global)
norm_swd       = _minmax_normalise(swd)
norm_coverage  = _minmax_normalise(coverage_l1)
norm_regime    = {lbl: _minmax_normalise(w1_regime[lbl]) for lbl in REGIME_LABELS}

tail_regime_score = {}
for n in model_names_w:
    tail_regime_score[n] = (
        TAIL_SCORE_WEIGHTS['W1 global flat'] * norm_w1_global[n]
        + TAIL_SCORE_WEIGHTS['Sliced W']     * norm_swd[n]
        + TAIL_SCORE_WEIGHTS['Low regime W1']  * norm_regime[REGIME_LABELS[0]][n]
        + TAIL_SCORE_WEIGHTS['Mid regime W1']  * norm_regime[REGIME_LABELS[1]][n]
        + TAIL_SCORE_WEIGHTS['High regime W1'] * norm_regime[REGIME_LABELS[2]][n]
        + TAIL_SCORE_WEIGHTS['Extreme W1']     * norm_regime[REGIME_LABELS[3]][n]
        + TAIL_SCORE_WEIGHTS['Coverage L1']    * norm_coverage[n]
    )

rank_tail = sorted(model_names_w, key=lambda n: tail_regime_score[n])

# Summary table
CW = 14
hdr = f"{'Metric':<24}" + ''.join(f"{n:>{CW}}" for n in model_names_w)
sep = '─' * len(hdr)
print(f"\n{hdr}\n{sep}")
print(f"{'W1 (global flat)':<24}" + ''.join(f"{w1_global[n]:>{CW}.5f}" for n in model_names_w))
print(f"{'Sliced W (surfaces)':<24}" + ''.join(f"{swd[n]:>{CW}.5f}"       for n in model_names_w))
print(sep)
for q in QUANTILE_LEVELS:
    print(f"{'W1  Q'+str(q):<24}" + ''.join(f"{w1_q[q][n]:>{CW}.5f}" for n in model_names_w))
print(sep)
for lbl in REGIME_LABELS:
    clean = lbl.replace('\n', ' ')
    print(f"{'W1  '+clean:<24}" + ''.join(f"{w1_regime_raw[lbl].get(n, np.nan):>{CW}.5f}" for n in model_names_w))
print(sep)
print(f"{'Coverage L1':<24}" + ''.join(f"{coverage_l1[n]:>{CW}.5f}" for n in model_names_w))
print(f"{'Tail/regime score':<24}" + ''.join(f"{tail_regime_score[n]:>{CW}.5f}" for n in model_names_w))

print("\nTail/regime-aware ranking, smaller is better:")
for k, n in enumerate(rank_tail, 1):
    print(f"  {k:>2}. {n:<28} score={tail_regime_score[n]:.5f}")

# Paper style 
mpl.rcParams.update(PAPER_RC)

x        = np.arange(len(model_names_w))
bar_clrs = [clr_cmp[n] for n in model_names_w]

# Fig 1: Global W1 + Sliced W
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
ax1.bar(x, [w1_global[n] for n in model_names_w], color=bar_clrs, alpha=0.85, width=0.55)
ax1.set_xticks(x); ax1.set_xticklabels(model_names_w, rotation=20, ha='right')
ax1.set_ylabel(r'$W_1$ (fraction)'); ax1.set_title(r'Global $W_1$  — flattened IV')

ax2.bar(x, [swd[n] for n in model_names_w], color=bar_clrs, alpha=0.85, width=0.55)
ax2.set_xticks(x); ax2.set_xticklabels(model_names_w, rotation=20, ha='right')
ax2.set_ylabel('SWD (fraction)'); ax2.set_title('Sliced Wasserstein  — surface vectors')

plt.tight_layout()
plt.savefig('figs/fig_wasserstein_global.pdf', bbox_inches='tight')
plt.show()

# Fig 2: W1 by quantile level
fig, ax = plt.subplots(figsize=(9, 5))
for n, arr in gen_models_w.items():
    vals = [w1_q[q][n] for q in QUANTILE_LEVELS]
    ax.plot(QUANTILE_LEVELS, vals, marker='o', linewidth=2.2,
            color=clr_cmp[n], linestyle=ls_cmp[n], label=n)
ax.set_xlabel('Quantile level (%)')
ax.set_ylabel(r'$W_1$ distance (pp)')
ax.set_title(r'$W_1$ on Quantile Surfaces vs Real Data')
ax.set_xticks(QUANTILE_LEVELS)
ax.legend()
plt.tight_layout()
plt.savefig('figs/fig_w1_quantiles.pdf', bbox_inches='tight')
plt.show()

# Fig 3: W1 by volatility regime 
n_reg   = len(REGIME_LABELS)
n_mod   = len(model_names_w)
w_bar   = 0.8 / n_mod
x_reg   = np.arange(n_reg)
offsets = (np.arange(n_mod) - (n_mod - 1) / 2) * w_bar

fig, ax = plt.subplots(figsize=(10, 5))
for j, (n, off) in enumerate(zip(model_names_w, offsets)):
    vals = [w1_regime_raw[lbl].get(n, np.nan) for lbl in REGIME_LABELS]
    ax.bar(x_reg + off, vals, width=w_bar * 0.9,
           color=clr_cmp[n], alpha=0.85, label=n)
ax.set_xticks(x_reg); ax.set_xticklabels(REGIME_LABELS)
ax.set_ylabel(r'$W_1$ distance (fraction)')
ax.set_title(r'$W_1$ by Volatility Regime vs Real Data')
ax.legend(ncol=2)
plt.tight_layout()
plt.savefig('figs/fig_w1_regimes.pdf', bbox_inches='tight')
plt.show()

# Fig 4: Integrated tail/regime-aware score 
fig, ax = plt.subplots(figsize=(10, 4.8))
score_vals = [tail_regime_score[n] for n in model_names_w]
ax.bar(x, score_vals, color=bar_clrs, alpha=0.88, width=0.58)
ax.set_xticks(x); ax.set_xticklabels(model_names_w, rotation=25, ha='right')
ax.set_ylabel('Normalised score')
ax.set_title('Integrated Tail/Regime-Aware Model Score')
ax.text(0.01, 0.97, 'Smaller is better', transform=ax.transAxes,
        va='top', ha='left', fontsize=PAPER_FONT_SIZE)
plt.tight_layout()
plt.savefig('figs/fig_tail_regime_score_spx.pdf', bbox_inches='tight')
plt.show()

# Fig 5: Regime coverage diagnostic
fig, ax = plt.subplots(figsize=(10, 5))
for j, (n, off) in enumerate(zip(model_names_w, offsets)):
    ax.bar(x_reg + off, regime_props[n], width=w_bar * 0.9,
           color=clr_cmp[n], alpha=0.85, label=n)
ax.plot(x_reg, real_regime_props, color='black', marker='o', linewidth=2.4,
        label='Real data')
ax.set_xticks(x_reg); ax.set_xticklabels(REGIME_LABELS)
ax.set_ylabel('Regime frequency')
ax.set_title('Generated Regime Coverage vs Real Data')
ax.legend(ncol=2)
plt.tight_layout()
plt.savefig('figs/fig_regime_coverage_spx.pdf', bbox_inches='tight')
plt.show()

mpl.rcParams.update(PAPER_RC)


In [ ]:
# W1 on explicit economic surface factors
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wasserstein_distance

os.makedirs("figs", exist_ok=True)

atm_idx = int(np.argmin(np.abs(np.asarray(m_new) - 1.0)))
left_idx = slice(None, NM // 4)
right_idx = slice(3 * NM // 4, None)
short_idx = slice(None, NT // 4)
long_idx = slice(3 * NT // 4, None)

print(f"ATM index={atm_idx}, m={m_new[atm_idx]:.4f}")
print(f"Short tenors={tau_new[short_idx]}")
print(f"Long tenors={tau_new[long_idx]}")


def surface_factors(arr):
    """Return level, skew, term-structure and curvature series for (N,T,M)."""
    arr = np.asarray(arr, dtype=np.float32)
    atm = arr[:, :, atm_idx]
    level = atm.mean(axis=1)
    term_structure = (
        atm[:, long_idx].mean(axis=1) - atm[:, short_idx].mean(axis=1)
    )
    left_wing = arr[:, :, left_idx].mean(axis=(1, 2))
    right_wing = arr[:, :, right_idx].mean(axis=(1, 2))
    skew = left_wing - right_wing
    curvature = 0.5 * (left_wing + right_wing) - level
    return {
        "Level": level,
        "Skew": skew,
        "Term Structure": term_structure,
        "Curvature": curvature,
    }


FACTOR_LABELS = ["Level", "Skew", "Term Structure", "Curvature"]
factors_all = {name: surface_factors(arr) for name, arr in models_cmp.items()}
real_factors = factors_all["Data (target)"]
gen_names = [name for name in models_cmp if name != "Data (target)"]

w1_factors = {
    name: {
        factor: wasserstein_distance(real_factors[factor], factors_all[name][factor])
        for factor in FACTOR_LABELS
    }
    for name in gen_names
}

factor_table = pd.DataFrame(w1_factors).T[FACTOR_LABELS]
display(factor_table.style.format("{:.6f}"))
factor_table.to_csv(RUN_DIR / "w1_explicit_factors.csv")

n_models = len(gen_names)
bar_width = 0.82 / max(n_models, 1)
x = np.arange(len(FACTOR_LABELS))
offsets = (np.arange(n_models) - (n_models - 1) / 2) * bar_width

fig, ax = plt.subplots(figsize=(14, 5.5))
for j, name in enumerate(gen_names):
    ax.bar(
        x + offsets[j],
        [w1_factors[name][factor] for factor in FACTOR_LABELS],
        width=bar_width * 0.9,
        color=clr_cmp[name],
        alpha=0.85,
        label=name,
    )
ax.set_xticks(x)
ax.set_xticklabels(FACTOR_LABELS)
ax.set_ylabel(r"$W_1$ distance (IV fraction)")
ax.set_title(r"$W_1$ on Explicit Volatility-Surface Factors")
ax.legend(ncol=3, fontsize=PAPER_FONT_SIZE)
plt.tight_layout()
plt.savefig("figs/w1_explicit_surface_factors.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# W1 on PCA scores fitted only on real surfaces 
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from scipy.stats import wasserstein_distance

os.makedirs("figs", exist_ok=True)

X_real = np.asarray(models_cmp["Data (target)"], dtype=np.float32)
X_real_flat = X_real.reshape(len(X_real), -1)
pca = PCA(n_components=4, random_state=SEED).fit(X_real_flat)
real_scores = pca.transform(X_real_flat)

PC_NAMES = ["PC1", "PC2", "PC3", "PC4"]
print("Explained-variance ratio:", np.round(pca.explained_variance_ratio_, 5))
print(
    "PC labels are intentionally kept as PC1--PC4: their economic meaning must "
    "be inferred from the loadings rather than assumed."
)

w1_pca = {}
for model_name, X_gen in models_cmp.items():
    if model_name == "Data (target)":
        continue
    gen_scores = pca.transform(np.asarray(X_gen).reshape(len(X_gen), -1))
    w1_pca[model_name] = [
        wasserstein_distance(real_scores[:, k], gen_scores[:, k])
        for k in range(4)
    ]

pca_table = pd.DataFrame(w1_pca, index=PC_NAMES).T
pca_table["Mean PC W1"] = pca_table.mean(axis=1)
display(pca_table.style.format("{:.6f}"))
pca_table.to_csv(RUN_DIR / "w1_pca_scores.csv")

model_names = list(w1_pca)
n_models = len(model_names)
bar_width = 0.82 / max(n_models, 1)
x = np.arange(4)
offsets = (np.arange(n_models) - (n_models - 1) / 2) * bar_width

fig, ax = plt.subplots(figsize=(14, 5.5))
for j, name in enumerate(model_names):
    ax.bar(
        x + offsets[j], w1_pca[name],
        width=bar_width * 0.9,
        color=clr_cmp[name], alpha=0.85, label=name,
    )
ax.set_xticks(x)
ax.set_xticklabels(PC_NAMES)
ax.set_ylabel(r"$W_1$ distance (PCA-score units)")
ax.set_title(r"$W_1$ on Real-Data PCA Scores")
ax.legend(ncol=3, fontsize=PAPER_FONT_SIZE)
plt.tight_layout()
plt.savefig("figs/w1_pca_scores.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# L-FM vs VolGAN vs Real 
import os
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

os.makedirs('figs', exist_ok=True)

mpl.rcParams.update(PAPER_RC)
sns.set_theme(context="paper", style="ticks", rc=PAPER_RC)
LW = 2.2

# L-FM and VolGAN
q_ref     = real_np_cmp.reshape(-1)          # reference distribution
gen_models = {'L-FM': l_fm_cmp, 'VolGAN': vg_np, 'DiffPen' : vd_np,}
clr_q      = {'L-FM': '#E07B00', 'VolGAN': '#9467BD', 'DiffPen' : '#1F77B4',}
ls_q       = {'L-FM': '--',      'VolGAN': ':', 'DiffPen' : '-.',}
mk_q       = {'L-FM': 'o',       'VolGAN': 's', 'DiffPen' : 'k',}

percs    = np.linspace(0, 100, 500)
q_real   = np.percentile(q_ref, percs)
safe_q   = np.where(np.abs(q_real) > 1e-6, q_real, np.nan)

# Fig 1: Q-Q plots
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
for ax, (name, arr) in zip(axes, gen_models.items()):
    q_model = np.percentile(arr.reshape(-1), percs)
    ax.scatter(q_real, q_model, s=8, color=clr_q[name], alpha=0.55,
               label=name, zorder=3)
    lo_lim = min(q_real.min(), q_model.min())
    hi_lim = max(q_real.max(), q_model.max())
    ax.plot([lo_lim, hi_lim], [lo_lim, hi_lim],
            color='0.35', linestyle='-', linewidth=1.5, label='Perfect fit', zorder=2)
    ax.set_xlabel('Real quantiles')
    ax.set_ylabel(f'{name} quantiles')
    ax.set_title(f'Q–Q Plot: {name} vs Real')
    ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig('figs/fig_qq.pdf', bbox_inches='tight')
plt.show()

# Fig 2a: Absolute quantile deviation 
fig, ax = plt.subplots(figsize=(8, 4.8))
for name, arr in gen_models.items():
    q_model = np.percentile(arr.reshape(-1), percs)
    ax.plot(percs, np.abs(q_model - q_real),
            color=clr_q[name], linewidth=LW, linestyle=ls_q[name], label=name)
ax.set_xlabel('Percentile')
ax.set_ylabel(r'$|Q_{\mathrm{model}}(p) - Q_{\mathrm{real}}(p)|$')
ax.set_title('Absolute Quantile Deviation')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('figs/fig_qdiff_abs.pdf', bbox_inches='tight')
plt.show()

# Fig 2b: Relative quantile deviation
fig, ax = plt.subplots(figsize=(8, 4.8))
for name, arr in gen_models.items():
    q_model = np.percentile(arr.reshape(-1), percs)
    rel_dev = np.abs(q_model - q_real) / np.abs(safe_q) * 100
    ax.plot(percs, rel_dev,
            color=clr_q[name], linewidth=LW, linestyle=ls_q[name], label=name)
ax.set_xlabel('Percentile')
ax.set_ylabel(r'Relative deviation (\%)')
ax.set_title('Relative Quantile Deviation')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('figs/fig_qdiff_rel.pdf', bbox_inches='tight')
plt.show()

mpl.rcParams.update(PAPER_RC)

key_percs = [1, 5, 10, 25, 50, 75, 90, 95, 99]
names     = list(gen_models.keys())

header = f"{'Pct':>5s}  {'Real':>9s}" + ''.join(f"  {n:>10s}  {'Δ':>8s}" for n in names)
print(header)
print('─' * len(header))
for p in key_percs:
    q_r = np.percentile(q_ref, p)
    row = f"{p:>4d}%  {q_r:9.4f}"
    for name, arr in gen_models.items():
        q_m = np.percentile(arr.reshape(-1), p)
        row += f"  {q_m:10.4f}  {q_m - q_r:+8.4f}"
    print(row)

print(f"\n{'Model':<12}  {'RMSE (abs)':>12}  {'MAE (abs)':>12}  {'RMSE (rel %)':>14}  {'MAE (rel %)':>13}")
print('─' * 70)
for name, arr in gen_models.items():
    q_m   = np.percentile(arr.reshape(-1), percs)
    diff  = q_m - q_real
    rel   = diff / np.abs(safe_q) * 100
    rmse  = np.sqrt(np.nanmean(diff**2))
    mae   = np.nanmean(np.abs(diff))
    rrmse = np.sqrt(np.nanmean(rel**2))
    rmae  = np.nanmean(np.abs(rel))
    print(f"{name:<12}  {rmse:12.5f}  {mae:12.5f}  {rrmse:14.2f}  {rmae:13.2f}")

In [ ]:
# Density & Smile Comparison + Arbitrage Errors
import seaborn as sns
from scipy.stats import norm
import matplotlib.pyplot as plt
import numpy as np

real_np        = real_iv[:, 0].cpu().numpy() / 100.0p
#real_np        = surf_train_n[:, 0].cpu().numpy() / 100.0 #Compare training or all or test
fm_trigo_np        = ivs_ldm_fm_trigo      / 100.0
fm_linear_np   = ivs_ldm_fm_linear_raw / 100.0
fm_quad_np     = ivs_ldm_fm_quad_raw   / 100.0
fm_vp_np       = ivs_ldm_fm_vp_raw     / 100.0

models = {
    'Real'       : real_np,
    'FM-trigo'   : fm_trigo_np,
    'FM-linear'  : fm_linear_np,
    'FM-quad'    : fm_quad_np,
    'FM-VP'      : fm_vp_np,
}
clr = {
    'Real'       : 'black',
    'FM-trigo'   : 'steelblue',
    'FM-linear'  : 'darkorange',
    'FM-quad'    : 'seagreen',
    'FM-VP'      : 'mediumpurple',
}

# Density comparison 
fig, ax = plt.subplots(figsize=(9, 5))
for name, arr in models.items():
    sns.kdeplot(arr.reshape(-1), ax=ax, label=name,
                color=clr[name], linewidth=2)
ax.set_xlim(-0.05, 0.6)
ax.set_xlabel('Implied Volatility'); ax.set_ylabel('Density')
ax.set_title('Distribution of Implied Volatility — All Models vs Real')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Summary statistics
print(f"\n{'Model':<14} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'Skew':>8}")
print('-' * 60)
for name, arr in models.items():
    x    = arr.reshape(-1)
    skew = ((x - x.mean())**3).mean() / x.std()**3
    print(f"{name:<14} {x.mean():8.4f} {x.std():8.4f} "
          f"{x.min():8.4f} {x.max():8.4f} {skew:8.4f}")

# Average Smile 
t_indices = [2, NT // 2, NT - 3]
t_labels  = [f'Short  τ={tau_new[i]:.2f}yr' for i in t_indices]
t_labels[1] = f'Mid  τ={tau_new[t_indices[1]]:.2f}yr'
t_labels[2] = f'Long  τ={tau_new[t_indices[2]]:.2f}yr'

fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
for ax, t_idx, t_lbl in zip(axes, t_indices, t_labels):
    for name, arr in models.items():
        ax.plot(m_new, arr[:, t_idx, :].mean(0),
                label=name, color=clr[name], linewidth=2)
    ax.set_title(t_lbl); ax.set_xlabel('Moneyness')
    ax.set_ylabel('Implied Volatility')
    ax.legend(fontsize=PAPER_FONT_SIZE); ax.grid(alpha=0.3)
fig.suptitle('Average Volatility Smile — All Models vs Real', fontsize=PAPER_FONT_SIZE, y=1.02)
plt.tight_layout(); plt.show()

# Average Skew 
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
for ax, t_idx, t_lbl in zip(axes, t_indices, t_labels):
    for name, arr in models.items():
        skews = np.gradient(arr[:, t_idx, :], m_new, axis=1)
        ax.plot(m_new, skews.mean(0), label=name,
                color=clr[name], linewidth=2)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_title(t_lbl); ax.set_xlabel('Moneyness')
    ax.set_ylabel('dIV / dK')
    ax.legend(fontsize=PAPER_FONT_SIZE); ax.grid(alpha=0.3)
fig.suptitle('Average Volatility Skew (dIV/dK) — All Models vs Real',
             fontsize=PAPER_FONT_SIZE, y=1.02)
plt.tight_layout(); plt.show()

# Arbitrage penalties
if "batch_phi" not in globals():
    raise RuntimeError("Run the pipeline-control/helper cell first.")
penalties = {name: batch_phi(arr, m_new, tau_new)
             for name, arr in models.items()}

model_names = list(models.keys())
COLORS = clr   # re-use same palette

keys   = ['phi',           'calendar',           'call',               'butterfly']
labels = [r'$\Phi$ total', r'$p_1$ calendar',    r'$p_2$ monotonicity', r'$p_3$ butterfly']

# Histograms
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, key, lbl in zip(axes, keys, labels):
    for name in model_names:
        ax.hist(penalties[name][key], bins=25, density=True, alpha=0.45,
                color=COLORS[name], label=name)
    ax.set_xlabel('Penalty'); ax.set_ylabel('Density')
    ax.set_title(lbl); ax.legend(fontsize=PAPER_FONT_SIZE)
fig.suptitle(r'No-Arbitrage Penalty $\Phi$ — Distribution Across Surfaces',
             fontsize=PAPER_FONT_SIZE)
plt.tight_layout(); plt.show()

# Boxplots
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, key, lbl in zip(axes, keys, labels):
    bp = ax.boxplot([penalties[n][key] for n in model_names],
                    patch_artist=True, notch=False,
                    medianprops=dict(color='black', lw=2))
    for patch, name in zip(bp['boxes'], model_names):
        patch.set_facecolor(COLORS[name]); patch.set_alpha(0.65)
    ax.set_xticks(range(1, len(model_names)+1))
    ax.set_xticklabels(model_names, rotation=20, ha='right')
    ax.set_ylabel('Penalty'); ax.set_title(lbl)
fig.suptitle('No-Arbitrage Penalties — Boxplot', fontsize=PAPER_FONT_SIZE)
plt.tight_layout(); plt.show()

# Stacked bar breakdown
fig, ax = plt.subplots(figsize=(9, 5))
x   = np.arange(len(model_names))
p1m = [penalties[n]['calendar'].mean()  for n in model_names]
p2m = [penalties[n]['call'].mean()      for n in model_names]
p3m = [penalties[n]['butterfly'].mean() for n in model_names]
ax.bar(x, p1m,                      color='#4878CF', alpha=0.85, label=r'$p_1$ calendar')
ax.bar(x, p2m, bottom=p1m,          color='#6ACC65', alpha=0.85, label=r'$p_2$ moneyness')
ax.bar(x, p3m, bottom=np.add(p1m, p2m),
                                     color='#D65F5F', alpha=0.85, label=r'$p_3$ butterfly')
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=PAPER_FONT_SIZE, rotation=15, ha='right')
ax.set_ylabel(r'Mean $\Phi$')
ax.set_title('Mean Penalty Breakdown by Component')
ax.legend(); plt.tight_layout(); plt.show()

# Summary table
col_w = 13
header = f"{'':16s}" + ''.join(f"{n:>{col_w}s}" for n in model_names)
print(f"\n{header}")
print('-' * (16 + col_w * len(model_names)))
for key, lbl in zip(keys, ['phi (total)', 'p1 calendar', 'p2 moneyness', 'p3 butterfly']):
    row = ''.join(f"{penalties[n][key].mean():>{col_w}.5f}" for n in model_names)
    print(f"{lbl:<16}{row}")
for name in model_names:
    phi = np.asarray(
        penalties[name]["phi"],
        dtype=float,
    )

    n_zero = int(
        np.sum(np.abs(phi) < ARB_TOL)
    )

    percentage = (
        100.0 * n_zero / len(phi)
    )

    print(
        f"{name:<30}"
        f" zero-arb: {n_zero}/{len(phi)}"
        f" ({percentage:5.1f}%)"
        f"   mean phi={phi.mean():.6f}"
        f"   max phi={phi.max():.6f}"
    )


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import norm, wasserstein_distance
from IPython.display import display

SAMPLING_SEEDS = (42, 43, 44, 45, 46)
N_GEN_SEED = int(globals().get("N_GEN_PER_SEED", N_GEN))

OUT = Path("data/treated/spx_full_multiseed")
FIG_OUT = Path("figs/spx_full_multiseed")

OUT.mkdir(parents=True, exist_ok=True)
FIG_OUT.mkdir(parents=True, exist_ok=True)

required = (
    "ae",
    "fm_objects",
    "latent_ddpm",
    "iv_ddpm",
    "vg_gen",
    "models_cmp",
)

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing trained objects in memory: {missing}. "
        "Do not restart the kernel."
    )

ae.save(
    str(OUT / "frozen_ivs_ae_spx.pt"),
    surf_lo=lo_v,
    surf_hi=hi_v,
)

for name, model in fm_objects.items():
    model.save(
        str(OUT / f"frozen_lfm_{name}_spx.pt")
    )

latent_ddpm.save(
    str(OUT / "frozen_latent_ddpm_spx.pt")
)

iv_ddpm.save(
    str(OUT / "frozen_iv_ddpm_spx.pt")
)

torch.save(
    {
        "generator_state_dict": {
            key: value.detach().cpu()
            for key, value in vg_gen.state_dict().items()
        },
        "config": dict(volgan_defaults),
        "lo_v": float(lo_v),
        "hi_v": float(hi_v),
        "m_new": np.asarray(m_new),
        "tau_new": np.asarray(tau_new),
    },
    OUT / "frozen_volgan_spx.pt",
)

print("Frozen in-memory models saved.")
print("No Optuna and no training will be rerun.")

fresh_names = {
    "L-FM linear",
    "L-FM quadratic",
    "L-FM VP",
    "L-FM trigo",
    "Latent DDPM",
    "Diffusion",
    "VolGAN",
    "VolGAN (raw)",
}

path_pools = {
    name: np.asarray(arr, dtype=np.float32)
    for name, arr in models_cmp.items()
    if (
        name != "Data (target)"
        and name not in fresh_names
    )
}

def phi_vectorised(surfaces):
    sigma = np.asarray(surfaces,dtype=np.float64)
    T, M = np.meshgrid(tau_new,m_new,indexing="ij")
    T = np.clip(T[None],1e-12,None)
    M = M[None]
    vol = np.clip(sigma,1e-8,None)
    d1 = (-M+ 0.5 * vol**2 * T) / (vol * np.sqrt(T))
    d2 = d1 - vol * np.sqrt(T)
    calls = (norm.cdf(d1)- np.exp(M) * norm.cdf(d2))
    dm = np.diff(m_new)
    dt = np.diff(tau_new)
    p_call = np.maximum(np.diff(calls, axis=2)/ dm[None, None, :],0.0).sum(axis=(1, 2))

    second_difference = (calls[:, :, 2:]- 2.0 * calls[:, :, 1:-1]+ calls[:, :, :-2])
    p_butterfly = np.maximum(-second_difference,0.0).sum(axis=(1, 2))
    p_calendar = np.maximum(-np.diff(calls, axis=1)/ dt[None, :, None],0.0).sum(axis=(1, 2))
    return (
        p_call
        + p_butterfly
        + p_calendar
    )

def gentle_volgan_reweight(raw_pct,seed,strength=0.35,n_bins=10):
    raw_pct = np.asarray(raw_pct,dtype=np.float32)
    raw_frac = _ensure_fraction_16x32(raw_pct,"VolGAN raw")
    phi = phi_vectorised(raw_frac)
    level = raw_pct.mean(axis=(1, 2))
    edges = np.unique(np.quantile(level,np.linspace(0.0,1.0,n_bins + 1)))

    if len(edges) <= 2:
        bins = np.zeros(len(level),dtype=int)
    else:
        bins = np.digitize(level,edges[1:-1],right=True)
    weights = np.ones(len(raw_pct),dtype=np.float64)

    for current_bin in np.unique(bins):
        idx = np.flatnonzero(bins == current_bin)
        if len(idx) < 2:
            continue

        order = np.argsort(phi[idx],kind="mergesort")
        ranks = np.empty(
            len(idx),
            dtype=np.float64,
        )

        ranks[order] = np.linspace(0.0,1.0,len(idx))
        local_weights = (1.0 + strength* (1.0 - 2.0 * ranks))
        local_weights /= local_weights.mean()
        weights[idx] = local_weights

    weights /= weights.sum()

    rng = np.random.default_rng(int(seed))
    selected = rng.choice(len(raw_pct),size=N_GEN_SEED,replace=True,p=weights)
    return raw_pct[selected]

def generate_from_seed(seed):
    generated = {}
    fm_labels = {
        "linear": "L-FM linear",
        "quadratic": "L-FM quadratic",
        "vp": "L-FM VP",
        "trigo": "L-FM trigo"}
    for offset, (path_name, label) in enumerate(fm_labels.items()):
        set_all_seeds(seed + 100 + offset)
        sample_pct = _decode(fm_objects[path_name].sample(N_GEN_SEED,device=device))
        generated[label] = (_ensure_fraction_16x32(sample_pct,label))

    set_all_seeds(seed + 200)

    latent_pct = _decode(latent_ddpm.sample(N_GEN_SEED,device=device))
    generated["Latent DDPM"] = (_ensure_fraction_16x32(latent_pct,"Latent DDPM"))
    set_all_seeds(seed + 300)
    diffusion_pct = iv_ddpm.sample(N_GEN_SEED,sampler="ddim",ddim_steps=int(ddpm_defaults["ddim_steps"]))
    generated["Diffusion"] = (_ensure_fraction_16x32(diffusion_pct,"Diffusion"))
    set_all_seeds(seed + 400)
    volgan_raw_pct = sample_volgan_diverse(gen=vg_gen,surf_train_n=surf_train_np,N_GEN=N_GEN_SEED,noise_dim=int(volgan_defaults["noise_dim"]),
        lo_v=lo_v,hi_v=hi_v,device=device,n_seeds=int(volgan_defaults["n_seeds"]))
    generated["VolGAN (raw)"] = (_ensure_fraction_16x32(volgan_raw_pct,"VolGAN (raw)"))
    volgan_reweighted_pct = (gentle_volgan_reweight(volgan_raw_pct,seed=seed + 500))

    generated["VolGAN"] = (_ensure_fraction_16x32(volgan_reweighted_pct,"VolGAN"))
    bootstrap_rng = np.random.default_rng(seed + 600)

    for name, pool in path_pools.items():
        selected = bootstrap_rng.choice(len(pool),size=N_GEN_SEED,replace=True)
        generated[name] = pool[selected]

    return generated

def metric_row(real,generated,seed,model):
    phi = phi_vectorised(generated)
    row = {
        "seed": int(seed),
        "model": model,
        "generation_mode": (
            "fresh frozen-model sampling"
            if model not in path_pools
            else "bootstrap of frozen path pool"
        ),
        "global_w1": float(
            wasserstein_distance(
                real.reshape(-1),
                generated.reshape(-1),
            )
        ),
        "sliced_w1": float(
            sliced_wasserstein(
                real.reshape(
                    len(real),
                    -1,
                ),
                generated.reshape(
                    len(generated),
                    -1,
                ),
                n_proj=300,
                seed=123456,
            )
        ),
        "mean_phi": float(
            phi.mean()
        ),
        "arb_free_pct": float(
            100.0
            * np.mean(
                np.abs(phi) < ARB_TOL
            )
        ),
        "generated_mean_iv": float(
            generated.mean()
        ),
        "generated_std_iv": float(
            generated.std()
        ),
    }

    for quantile in (5,50,95,99):
        real_surface = np.percentile(real,quantile,axis=0)
        generated_surface = np.percentile(generated,quantile,axis=0)
        row[f"q{quantile:02d}_surface_error"] = float(np.mean(np.abs(real_surface- generated_surface)))
    return row


real_sampling = np.asarray(models_cmp["Data (target)"],dtype=np.float32)
sampling_rows = []

for sampling_seed in SAMPLING_SEEDS:
    print(
        f"Generating with seed "
        f"{sampling_seed}..."
    )

    generated_models = generate_from_seed(
        sampling_seed
    )

    for model_name, generated in (
        generated_models.items()
    ):
        sampling_rows.append(
            metric_row(
                real=real_sampling,
                generated=generated,
                seed=sampling_seed,
                model=model_name,
            )
        )

    del generated_models

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


sampling_trials = pd.DataFrame(
    sampling_rows
)

metric_columns = [
    "global_w1",
    "sliced_w1",
    "q05_surface_error",
    "q50_surface_error",
    "q95_surface_error",
    "q99_surface_error",
    "mean_phi",
    "arb_free_pct",
    "generated_mean_iv",
    "generated_std_iv",
]

sampling_summary = (
    sampling_trials
    .groupby(
        [
            "model",
            "generation_mode",
        ]
    )[metric_columns]
    .agg(
        [
            "mean",
            "std",
        ]
    )
)

sampling_summary.columns = [
    f"{metric}_{stat}"
    for metric, stat
    in sampling_summary.columns
]

sampling_summary = (
    sampling_summary
    .reset_index()
    .sort_values(
        "sliced_w1_mean"
    )
    .reset_index(
        drop=True
    )
)

sampling_trials.to_csv(
    OUT / "sampling_seed_trials.csv",
    index=False,
)

sampling_summary.to_csv(
    OUT / "sampling_seed_summary.csv",
    index=False,
)

print(
    "\nMean ± standard deviation across "
    "generation seeds from the same frozen models:"
)

display(sampling_summary)


# ============================================================
# 7. ACM 9 PT FIGURES
# ============================================================

x = np.arange(
    len(sampling_summary)
)

labels = sampling_summary["model"]


fig, ax = plt.subplots(
    figsize=(9.0, 4.8)
)

ax.errorbar(
    x,
    sampling_summary["global_w1_mean"],
    yerr=sampling_summary["global_w1_std"],
    fmt="o",
    capsize=3,
    label=r"Global $W_1$",
)

ax.errorbar(
    x,
    sampling_summary["sliced_w1_mean"],
    yerr=sampling_summary["sliced_w1_std"],
    fmt="s",
    capsize=3,
    label=r"Sliced $W_1$",
)

ax.set_xticks(x)

ax.set_xticklabels(
    labels,
    rotation=40,
    ha="right",
)

ax.set_ylabel("Distance")

ax.set_title(
    "Frozen Models Across Five Generation Seeds"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    FIG_OUT
    / "sampling_seed_w1_mean_std.pdf",
    bbox_inches="tight",
)

plt.show()


fig, ax = plt.subplots(
    figsize=(9.0, 4.8)
)

ax.errorbar(
    x,
    sampling_summary["arb_free_pct_mean"],
    yerr=sampling_summary["arb_free_pct_std"],
    fmt="o",
    capsize=3,
    label="Arbitrage-free rate",
)

ax.set_xticks(x)

ax.set_xticklabels(
    labels,
    rotation=40,
    ha="right",
)

ax.set_ylabel(
    "Arbitrage-free surfaces (%)"
)

ax.set_title(
    "Arbitrage-Free Rate Across Five Generation Seeds"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    FIG_OUT
    / "sampling_seed_arb_mean_std.pdf",
    bbox_inches="tight",
)

plt.show()